# Evaluation - Charge-Light Matching - AFTER Beam Window Cut

Same multi-file evaluation as
`Evaluation_ChargeLightMatching_BeforeBeamWindowCut.ipynb`, with the
**beam-window cut applied to the reco side**: only clustering-global clusters
whose bridged flash time falls inside
`[BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US]` = [0.33, 1.93] us survive into
efficiency, purity, matching, metadata and every plot. The in-spill reco
population is neutrino-dominated, so this is the notebook for questions about
how well **neutrino** clusters are reconstructed.

The cut is deliberately reco-side only. "In beam window" is not a truth
quantity -- true clusters carry no flash and no time -- so a true-side version
could only be inferred by matching to a beam-window-flashed reco cluster,
folding beam timing into what would read as a truth-level selection. The true
side therefore still contains its cosmic clusters; a cosmic true cluster that
now goes unmatched means "no in-spill reco cluster near it", which is the
intended reading rather than a bug.

Set `Apply_beam_window_cut = False` in the configuration cell to reproduce the
before-cut notebook exactly. Output goes to
`multi_file_plots_charge_light_matching/Evaluation_After_TimeWindowCut_TrueNeutrinos/`, a
subdirectory of the shared charge-light tree, so the two notebooks never
overwrite each other.

## Three output roots: all / in-volume / out-of-volume true neutrinos

Each run writes the same familiar tree (`file0/event_XXX/`, `file0/file_summary/`,
`job_summary/`) three times, once per true-cluster population:

| Root | True clusters it holds |
|---|---|
| `All_True_Clusters/` | everything -- cosmic and neutrino, both vertex volumes. This is the output this notebook produced before the split. |
| `In_Volume_True_Neutrinos/` | true neutrinos whose `mc.json` interaction vertex is **inside** the wire-readout sensitive box |
| `Out_Volume_True_Neutrinos/` | true neutrinos whose vertex is **outside** it |

**The evaluation runs once.** Cuts, `cluster_category`, `EvaluateEfficiency`,
`EvaluatePurity`, 1-to-1 / 1-to-many matching and metadata are computed a single
time per event against the full true and reco populations; the three roots are
three *renderings* of those same finished records, filtered by
`metadata.build_neutrino_volume_map` / `filter_records_by_volume` (the join is
exact: a neutrino interaction's true cluster id is `99990 + nu_idx`, which is the
`true_cluster_id` every record carries). An efficiency or purity value is the same
number in every root it appears in, because it *is* the same number.

**Only the true side is filtered -- the reco side is never cut.** Every reco
cluster that passed the beam-window cut is in play in all three roots, which is
what makes the split worth reading: the purity attached to an in-volume neutrino
still measures the cosmic contamination of the reco cluster it matched.

What that means in the In/Out roots:

- **cosmic true clusters are absent** (they have no interaction vertex, so no
  volume); cosmic *reco* clusters are fully present, as matches and as
  contamination
- **unmatched reco clusters are absent** -- `EvaluatePurity`'s
  `true_cluster_id = 8888` rows describe reco clusters that touched no true
  cluster at all, so they belong to neither volume. They stay in
  `All_True_Clusters/`
- **unmatched true neutrinos are kept**, at efficiency 0: their
  `EvaluateEfficiency` row is keyed by their own cluster id (only
  `reco_cluster_id` is the sentinel), so a neutrino that reconstructed to nothing
  stays in the efficiency denominator -- which is the point of measuring the two
  volumes apart
- **no energy re-cut is needed**: `apply_energy_cutoff` (100 MeV) already ran
  before any of these records existed, so every true neutrino here is above
  threshold by construction. The ones it removed appear only in
  `removed_true_neutrino_info.txt`

**Reco-side diagnostics are drawn once**, in `All_True_Clusters/`: a flash and a
reco cluster carry no truth label, so `imaging_details/`, `clustering_details/`
and `reco_cluster_info.txt` have no in/out-of-volume version. The In/Out roots
carry a `README.txt` saying so.

Plot filenames are identical across roots, so the same plot can be diffed
root-to-root; titles carry the population where the drawer accepts a label.

Two more things sit beside the three roots:

- `In_vs_Out_Volume_True_Neutrinos/` -- the in-volume and out-of-volume curves
  overlaid on one canvas: clusteringlevel 1D efficiency vs true energy, and 1D
  purity vs reco charge. **Job level only** (an event or a file holds one or two
  interactions, which is not a curve), and drawn on **shared bins and a shared
  axis** derived from both populations together, since each root otherwise bins
  itself to its own range and the two would only look comparable. Efficiency uses
  the same population as `DrawClusterEfficiencyVsTrueEnergyPerJob` -- matched
  pairs plus unmatched true clusters at efficiency 0; purity uses the matched
  pairs only, an unmatched neutrino having no reco cluster to be pure.
- a top-level `summary.txt` -- one table of what each root holds and how many
  wall-clock seconds each took to render. The evaluation runs once, before any
  population loop, so it is charged to no population and the three render times do
  not sum to the job runtime; the difference is reported as "shared (read +
  evaluate)".

---


Evaluation for the **charge-light matching** JSON file format (combined-APA:
`img-global` reco clusters, `sed-sce_drift_smear_readout` true clusters, `mc`
particle truth tree, `op` optical/light info). This is a new, separate JSON
format/layout from the one `Evaluation_BeforeChargeLightMatching_BeforeBeamWindowCut.ipynb` reads —
that notebook and the modules it calls (`readfiles.py`, `selections.py`,
`efficiency_purity_estimate.py`, `efficiency_purity_draw.py`, etc.) are left
completely unchanged; the charge-light readers were added additively
alongside the existing ones in `readfiles.py`.

**Current stage**: this notebook unzips the test data (once) and reads the
four charge-light files above for each file/event, printing sanity-check
counts. The downstream selections / efficiency / purity / drawing pipeline
is not wired in yet — that comes once the cluster-ID / neutrino-identification
mapping for this format is defined.

In [1]:
# Charge-light matching is a combined-APA evaluation (img-global / sed-sce are
# already global across APAs) -- no per-APA/face looping like the older pipeline.
files     = "all"   # "all", or 1/2/3/... to limit the number of file subdirectories processed
events    = "all"   # "all", or 1/2/3/... to limit the number of events processed per file

# ========================================================================
# SELECTIVE FILE/EVENT FILTERING (Optional)
# ========================================================================
# Set to None to process all files/events, or specify to run only specific ones
# Example: target_file = "file0", target_event = 3  (to process only file0, event 3)
target_file  = None  # Set to "file0", "file1", etc. to process specific file only
target_event = None   # Set to a SINGLE event number (0, 1, ..., 9); use target_event_range below for a span

# Range of event numbers to run, inclusive on both ends: (1, 5) runs events
# 1,2,3,4,5. None runs every event. Applied on top of target_event, so leave
# target_event = None when using a range. This exists because target_event only
# ever compares equal to one int -- handing it a tuple silently matched nothing
# and produced a run with zero events and no plots.
target_event_range = None   # e.g. (1, 5) for events 1..5

# Range of file INDICES to run, inclusive on both ends: (6, 9) runs file6, file7,
# file8, file9. None runs every file. Matched on the number at the end of the
# directory name, NOT on position in the list -- the directories sort
# lexicographically (file0, file1, file10, file11, file2, ...), so a positional
# slice would pick the wrong files. This is the only file selector that survives
# that sort order; the `files = N` knob above still takes the first N in
# lexicographic order.
# NOTE: target_file (above) is applied too, so set it to None when using a range,
# otherwise only the one file that satisfies both runs.
target_file_range = None   # e.g. (6, 9) for file6..file9

# Fail fast rather than silently processing nothing: `evt != target_event` can
# never be False for a tuple, so target_event = (1, 5) used to skip every event
# and still write an empty summary.
if isinstance(target_event, (tuple, list)):
    raise ValueError(
        f"target_event={target_event} is a range, but target_event takes a single event number. "
        f"Use target_event_range={tuple(target_event)} and target_event = None instead.")


# ========================================================================
# Decide which plots to draw
# ========================================================================
b_draw_event_level_plots = True   # Draw event-level plots (one per event)
b_draw_file_level_plots  = False   # Draw file-level plots (one per file)
b_draw_job_level_plots   = True    # Draw job-level plots (one per job)

In [2]:
%load_ext autoreload
%autoreload 2

# python libraries
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from pathlib import Path
import sys
import os
from scipy.spatial import KDTree
import pandas as pd
import seaborn as sns
import time
from datetime import datetime

np.set_printoptions(linewidth=1000)

# Record job start time (used to report total job runtime at the end)
job_start_time = time.time()
print(f"Job started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


Job started at: 2026-08-07 14:19:48


In [3]:
# Import functions from Python modules
# Full pipeline now wired in: selections, cluster_category, efficiency, purity,
# 1-to-1/1-to-many matching, metadata, and all drawing. reassign_cluster_ID_true_charge_light
# IS used -- true clusters are grouped under 99990+nu_idx (99991, 99992, ... one
# per neutrino interaction) / avg-X (cosmic); see that function's docstring in
# selections.py for why this is a separate function from the legacy
# reassign_cluster_ID_true (still 9999-merging, still used by other pipelines).
# reassign_cluster_ID_reco IS ALSO used -- reco (clustering-global) clusters
# are relabeled by avg-X, same convention.
from readfiles import ensure_data_extracted, read_charge_light_files_for_event, flatten_mc_tree
from selections import (
    GroupClustersByID, build_true_points_charge_light, apply_deadarea_cut_true_charge_light,
    reassign_cluster_ID_true_charge_light, reassign_cluster_ID_reco,
    apply_energy_cutoff, apply_min_true_points_cutoff, apply_min_reco_points_cutoff,
    apply_wire_readout_sensitive_yz_plane_cut_true, apply_wire_readout_sensitive_yz_plane_cut_reco,
)
from cluster_category import cluster_category
from efficiency_purity_estimate import EvaluateEfficiency, EvaluatePurity
from clusterpairmatching import MatchTrueToReco1to1, MatchTruetoReco_OneToMany
from metadata import (
    build_cluster_flash_metadata, build_img_cluster_flash_metadata, CATHODE_CROSSING_TIME_DIFF_MAX_US,
    add_metadata_true_clusters, add_metadata_true_reco_pair_cluster,
    build_true_cluster_type_records, build_neutrino_vertex_records,
    # The two population splits. build_neutrino_volume_map turns the vertex
    # records into a {(event, true cluster id) -> 'in'|'out'} lookup and
    # build_neutrino_channel_map into a {... -> 'numu_CC'|'nue_CC'|'NC'} one;
    # filter_records_by_label then selects one population's rows out of any
    # already-computed record list, the same operation for either map. Nothing is
    # recomputed -- see the rendering cell below.
    build_neutrino_volume_map, build_neutrino_channel_map, filter_records_by_label, restrict_label_map,
)
# Information-file writers live in writeinformation.py, not metadata.py: metadata
# builds the in-memory records, this writes the human-readable .txt tables.
# write_neutrino_vertex_info / write_removed_neutrino_info are imported again: the
# job-level true_neutrino_info.txt and removed_true_neutrino_info.txt are written
# HERE as well as in SelectionAnalysis.ipynb. Same writers, same filenames, fed the
# same records -- this notebook's copies describe the population its own efficiency
# and purity numbers were computed on, so the two are read together rather than
# having to cross-reference the other notebook's output tree.
from writeinformation import (
    write_true_cluster_info, write_reco_cluster_info,
    write_neutrino_vertex_info, write_removed_neutrino_info,
    write_efficiency_info, write_pair_efficiency_info, write_purity_info,
)
from efficiency_purity_draw import (
    plot_efficiency_heatmap, plot_purity_heatmap,
    DrawEfficiencyVsTrueEnergyPerEvent, DrawEfficiencyVsTrueEnergyPerFile, DrawEfficiencyVsTrueEnergyPerJob,
    DrawClusterEfficiencyVsTrueEnergyPerEvent, DrawClusterEfficiencyVsTrueEnergyPerFile, DrawClusterEfficiencyVsTrueEnergyPerJob,
    DrawEfficiencyVsTrueEnergy_MatchedPairs_PerEvent, DrawEfficiencyVsTrueEnergy_MatchedPairs_PerFile, DrawEfficiencyVsTrueEnergy_MatchedPairs_PerJob,
    DrawPurityVsRecoChargePerEvent, DrawPurityVsRecoChargePerFile, DrawPurityVsRecoChargePerJob,
    DrawEfficiencyVsPurity_MatchedPairs,
    summarize_cluster_efficiency_by_energy, format_cluster_efficiency_by_energy,
    # Several populations on one canvas (job level only) -- the only plots where
    # the populations of a split meet; every other plot renders each population
    # into its own directory. Used twice: in-volume vs out-of-volume, and the
    # numu CC / nue CC / NC channels (CHANNEL_COMPARISON_STYLES).
    DrawClusterEfficiencyVsTrueEnergy_PopulationComparison, DrawPurityVsRecoCharge_PopulationComparison,
    CHANNEL_COMPARISON_STYLES,
)
from DrawRecoTrueClusters import (
    DrawTrueRecoClustersXZ, DrawTrueRecoClustersYZ, DrawTrueRecoClustersXY, DrawLabels, DrawNeutrinoRecoClusters,
    DrawTrueClusterWithMatchedReco, DrawTrueRecoMatchMultiplicity, DrawLabelPerFile,
    draw_clustering_global_clusters, DrawLabelsAggregated, DrawLabelsByNuIdx,
    DrawNeutrinoVertices, DrawNeutrinoVolumeCategory, DrawNeutrinoFlavor, DrawNeutrinoBreakdown,
    _draw_cosmic_category_bar,
)
from DrawRecoTrueFlashes import (
    draw_flashes, draw_img_global_clusters, draw_clustering_flashes, draw_cluster_flash_time_bar,
    BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US,
)

In [4]:
# Configuration: Parent directory containing multiple file subdirectories (file0/, file1/, ...)
#
# Expected structure (per file subdirectory). The preprocessed tree has no zip --
# its data/ is already present, so ensure_data_extracted() below simply no-ops.
# PARENT_DIR/
#   file0/data/0/0-img-global.json                      (reco clusters, combined APA)
#   file0/data/0/0-sed-sce_drift_smear_readout.json      (true clusters, combined APA)
#   file0/data/0/0-mc.json                               (particle truth ancestry tree)
#   file0/data/0/0-op.json                               (optical/light info)
#   file0/data/1/, 2/, ... (one subdirectory per event)
#   file1/mabc.zip, file1/data/..., etc.

# The DEAD-AREA-PREPROCESSED tree, produced once by preprocess_deadarea_cut.py.
# Its true-point files already have the dead-area cut applied, which is why
# Apply_deadarea_cut is False below -- see the note there. Read that script's
# docstring, or DEADAREA_PREPROCESSING.txt inside the tree, before switching this
# back to the raw tree: the two are NOT interchangeable.
PARENT_DIR = Path("Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut")

# Number of files to process (convert 'files' variable to num_files_to_process)
num_files_to_process = None if files == "all" else files

# Number of events to process (convert 'events' variable to num_events_to_process)
num_events_to_process = None if events == "all" else events

# Output directory for plots: a subdirectory of the shared charge-light tree, NOT
# the tree root. The before-cut notebook writes its filenames straight into the
# root, so sharing that level would have whichever notebook ran last silently
# overwrite the other's plots and summaries.
PLOTBASEDIR = Path("multi_file_plots_charge_light_matching/Evaluation_After_TimeWindowCut_TrueNeutrinos")
PLOTBASEDIR.mkdir(parents=True, exist_ok=True)

print("Configuration:")
print(f"Parent directory: {PARENT_DIR}")
print(f"Plot base directory: {PLOTBASEDIR}")
print(f"Files to process: {files}")
print(f"Events to process: {events}")

if target_file is not None or target_event is not None or target_file_range is not None or target_event_range is not None:
    print(f"\n⚡ SELECTIVE FILTERING ENABLED:")
    print(f"  Target file: {target_file if target_file else 'all'}")
    print(f"  Target event: {target_event if target_event is not None else 'all'}")
    if target_event_range is not None:
        print(f"  Target event range: event{target_event_range[0]}..event{target_event_range[1]} (inclusive)")
    if target_file_range is not None:
        print(f"  Target file range: file{target_file_range[0]}..file{target_file_range[1]} (inclusive)")

# ========================================================================
# SELECTION PARAMETERS
# ========================================================================
# Matching/efficiency/purity radii and the geometry-based cuts (fiducial YZ box,
# dead-area) are reused at the SAME values as the existing pipeline -- detector
# geometry hasn't changed and these are unit-independent of the new q/energy field.
radius_efficiency         = 2
radius_purity_xz          = 2
radius_purity_yz          = 5
radius_purity_xy          = 5
min_recopoints_threshold  = 5

# min_true_points_cutoff / min_reco_points_cutoff are DISABLED for now: this
# format's point clouds are much sparser than the old imaging-based
# reconstruction -- real neutrino clusters have been seen with as few as 13
# points -- so the old threshold (200) would delete real signal clusters
# outright. Revisit once correct values are known for this format's point
# density.
#
# min_cluster_energy IS applied (Apply_energy_cutoff = True): sed-sce's
# per-point 'e' field (MeV) is a genuine energy deposit -- same physical
# quantity/units as the old (non charge-light) pipeline's energy column --
# so the old threshold (100 MeV) carries over directly. See
# build_true_points_charge_light's energy= parameter (falls back to 'q'
# for older-format files that lack 'e').
min_cluster_energy        = 100     # APPLIED (Apply_energy_cutoff = True below)
min_true_points_cutoff    = 200     # NOT APPLIED (Apply_min_true_points_cutoff = False below)
min_reco_points_cutoff    = 200     # NOT APPLIED (Apply_min_reco_points_cutoff = False below)

# Apply selections
Apply_energy_cutoff                         = True    # sed-sce's 'e' field is genuine MeV -- see note above
Apply_min_true_points_cutoff                = False   # disabled -- see note above
Apply_min_reco_points_cutoff                = False   # disabled -- see note above
Apply_wire_readout_sensitive_xz_plane_cut   = True
Apply_time_window_cut                       = False   # must stay disabled -- no per-point true time in this format
# BEAM-WINDOW (time) CUT -- the one cut that makes this the "after beam window
# cut" notebook. Keeps only reco clusters whose bridged flash time lies inside
# [BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US] = [0.33, 1.93] us, i.e. the in-spill
# population, which is neutrino-dominated. Set False to reproduce
# Evaluation_ChargeLightMatching_BeforeBeamWindowCut.ipynb exactly.
#
# This is NOT the same cut as Apply_time_window_cut above: that one needs a
# per-point TRUE time, which this format does not carry. The beam window is
# measured where the timing actually lives -- on the reco side, via the flash.
Apply_beam_window_cut                       = True
# The dead-area cut is APPLIED, just not here: PARENT_DIR above is the tree
# preprocess_deadarea_cut.py already cut. Applying it again would be a no-op that
# costs a polygon test per event. It is applied FIRST, before the energy cut,
# because reco points are never reconstructed inside a dead channel region -- true
# deposits there could never have been seen, so removing them is a correction that
# puts truth and reco on the same measurable volume, not a selection to rank
# alongside the others. Set this True only if you point PARENT_DIR back at a raw
# tree, and note that doing so restores the OLD ordering (dead area last).
Apply_deadarea_cut                          = False

# YZ Sensitivity cut parameters (detector geometry, unit: cm)
x_min = -250.0
x_max = 250.0
y_min = -200.0
y_max = 200.0
z_min = 0.15
z_max = 500.85

# Metadata label only (add_metadata_true_clusters/add_metadata_true_reco_pair_cluster
# store 'view' as a plain string field, not used for any logic) -- there's no
# 2-view/3-view distinction in the charge-light format, so this is just a constant.
view = "combined"

marker_size = 1

print("\nCuts applied:")
if Apply_energy_cutoff:
    print(f"- Energy cutoff applied (threshold {min_cluster_energy} MeV, using sed-sce's per-point 'e' field)")
if Apply_wire_readout_sensitive_xz_plane_cut:
    print(f"- Wire readout sensitive xz plane cut applied")
if Apply_beam_window_cut:
    print(f"- Beam window cut applied to RECO clusters only ({BEAM_WINDOW_MIN_US} - {BEAM_WINDOW_MAX_US} us flash time)")
else:
    print(f"- Beam window cut NOT applied -- this run reproduces the before-beam-window-cut notebook")
if Apply_deadarea_cut:
    print(f"- Dead area cut applied HERE (split by X sign into APA0/APA1, see apply_deadarea_cut_true_charge_light)")
else:
    print(f"- Dead area cut applied UPSTREAM by preprocess_deadarea_cut.py (baked into PARENT_DIR, before every other cut)")
print("\nCuts not applied (see note above)")
if not Apply_min_true_points_cutoff:
    print(f"- Minimum true points cutoff not applied (threshold {min_true_points_cutoff} too aggressive for this format's point density)")
if not Apply_min_reco_points_cutoff:
    print(f"- Minimum reco points cutoff not applied (threshold {min_reco_points_cutoff} too aggressive for this format's point density)")

# ========================================================================
# ONE-TIME EXTRACTION
# ========================================================================
# Each file subdirectory ships as a single zip file. ensure_data_extracted()
# only unzips if that file's data/ folder doesn't already exist yet, so
# re-running this notebook never re-extracts.
if PARENT_DIR.exists():
    for subdir in sorted(PARENT_DIR.iterdir()):
        if subdir.is_dir():
            ensure_data_extracted(subdir)
else:
    print(f"Error: Parent directory {PARENT_DIR} does not exist")


Configuration:
Parent directory: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut
Plot base directory: multi_file_plots_charge_light_matching/Evaluation_After_TimeWindowCut_TrueNeutrinos
Files to process: all
Events to process: all

Cuts applied:
- Energy cutoff applied (threshold 100 MeV, using sed-sce's per-point 'e' field)
- Wire readout sensitive xz plane cut applied
- Beam window cut applied to RECO clusters only (0.33 - 1.93 us flash time)
- Dead area cut applied UPSTREAM by preprocess_deadarea_cut.py (baked into PARENT_DIR, before every other cut)

Cuts not applied (see note above)
- Minimum true points cutoff not applied (threshold 200 too aggressive for this format's point density)
- Minimum reco points cutoff not applied (threshold 200 too aggressive for this format's point density)


In [5]:
def find_all_input_directories(parent_dir):
    """
    Scan parent directory for all subdirectories containing 'data' folder.
    Returns a list of file directories (file0/, file1/, etc.).
    """
    parent_dir = Path(parent_dir)
    if not parent_dir.exists():
        print(f"Error: Parent directory {parent_dir} does not exist")
        return []

    data_dirs = []
    for subdir in sorted(parent_dir.iterdir()):
        if subdir.is_dir():
            data_path = subdir / "data"
            if data_path.exists() and data_path.is_dir():
                data_dirs.append(subdir)
                print(f"Found: {subdir}")

    return data_dirs

def file_index_from_name(name):
    """
    Trailing integer of a file directory name ("file10" -> 10), or None if it
    has no trailing digits. Used by target_file_range so files are selected by
    their real index rather than by position in the lexicographically sorted
    list (file0, file1, file10, file11, file2, ...).
    """
    digits = ""
    for ch in reversed(name):
        if not ch.isdigit():
            break
        digits = ch + digits
    return int(digits) if digits else None


def detect_events_in_directory(input_dir):
    """
    Auto-detect the number of events in a directory.
    Events are identified as numeric subdirectories in data/.
    Returns a sorted list of event numbers.
    """
    input_dir = Path(input_dir)
    data_dir = input_dir / "data"

    if not data_dir.exists():
        print(f"Warning: Data directory {data_dir} does not exist")
        return []

    events = []
    for item in data_dir.iterdir():
        if item.is_dir():
            try:
                event_num = int(item.name)
                events.append(event_num)
            except ValueError:
                pass

    return sorted(events)

# Auto-detect all input directories from parent directory
print(f"Scanning parent directory: {PARENT_DIR}")
print("-" * 60)
input_directories = find_all_input_directories(PARENT_DIR)
# Limit to num_files_to_process files for testing
if num_files_to_process is not None:
    input_directories = input_directories[:num_files_to_process]
else:
    num_files_to_process = len(input_directories)
print("-" * 60)

print(f"\nFound {len(input_directories)} input directories with data/\n")
if input_directories:
    for input_dir in input_directories:
        detected_events = detect_events_in_directory(input_dir)
        if detected_events:
            print(f"  {input_dir.name}/data/: {len(detected_events)} events ({min(detected_events)}-{max(detected_events)})")
        else:
            print(f"  {input_dir.name}/data/: No events found")
else:
    print(f"Error: No subdirectories with 'data/' found in {PARENT_DIR}")


Scanning parent directory: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut
------------------------------------------------------------
Found: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file0
Found: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file1
Found: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file10
Found: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file11
Found: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file2
Found: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file3
Found: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file4
Found: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file5
Found: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file6
Fou

In [ ]:
# ============================================================================
# PER-POPULATION RENDERING -- the drawing/writing half of the evaluation
# ============================================================================
# The evaluation itself runs ONCE per event in the main loop below: cuts,
# cluster_category, EvaluateEfficiency, EvaluatePurity, 1-to-1 and 1-to-many
# matching, metadata -- all against the FULL true and reco populations of the
# event. The two functions here are the other half: they take those finished
# records and render one output directory from them. Handing them a FILTERED copy
# of the same records is what produces the in-volume / out-of-volume roots. No
# evaluation is repeated and no efficiency or purity number is recomputed --
# every value drawn in every root is the one computed in that single pass.
#
# ONLY THE TRUE SIDE IS EVER FILTERED. Every reco cluster that survived the beam
# window cut stays in play in all three roots, which is what makes the split
# meaningful: the purity attached to an in-volume neutrino still measures the
# cosmic contamination in the reco cluster it matched. What differs between roots
# is only WHICH TRUE CLUSTERS are shown -- all of them, the in-volume neutrinos,
# or the out-of-volume ones.
#
# Consequences of filtering by metadata.build_neutrino_volume_map, all intended:
#   - cosmic TRUE clusters vanish from the In/Out roots (no mc.json interaction,
#     hence no vertex record, hence no volume). Cosmic RECO clusters do not.
#   - EvaluatePurity's unmatched-reco rows (true_cluster_id=8888) vanish too --
#     they describe reco clusters that touched no true cluster at all, so they
#     belong to neither volume. They stay in the All root.
#   - an unmatched true NEUTRINO is KEPT (its EvaluateEfficiency row is keyed by
#     its own cluster id; only reco_cluster_id is the 8888 sentinel), so
#     neutrinos that reconstructed to nothing stay in the efficiency denominator
#     at efficiency 0. That is the number the split exists to measure.
#
# Reco-side-only diagnostics -- optical flashes, img-global/clustering-global
# cluster drawings, reco_cluster_info.txt -- are NOT rendered here. A flash is
# neither in-volume nor out-of-volume, so those are drawn once, into the All root,
# by the main loop.
#
# Both functions read the notebook-level geometry bounds (x_min..z_max) and
# min_cluster_energy as globals, exactly as the inline code they replace did.

# The three output roots. 'label' is appended to plot TITLES only, so the same plot
# has the same filename in every root and can be diffed directly. Most drawers put
# neither file_name nor level_name in a filename; DrawEfficiencyVsPurity_MatchedPairs
# does put level_name there, which is what its filename_level argument is for --
# the decorated level goes to the title, the plain one to the filename.
POPULATIONS = [
    {'key': 'all', 'dirname': 'All_True_Clusters',
     'label': None,
     'include_cosmic_plots': True},
    {'key': 'in',  'dirname': 'In_Volume_True_Neutrinos',
     'label': 'true neutrinos, vertex in volume',
     'include_cosmic_plots': False},
    {'key': 'out', 'dirname': 'Out_Volume_True_Neutrinos',
     'label': 'true neutrinos, vertex out of volume',
     'include_cosmic_plots': False},
]

# The SECOND split, by interaction channel, and independent of the volume one:
# what came out of the interaction rather than where it happened. Rendered by the
# same two functions from the same records, into
# <All root>/.../By_Interaction_Channel/<channel>/ at every level -- inside the
# All root because that root is the complete true population, so the three
# channels partition its neutrinos exactly.
#
# Channel comes from metadata.classify_neutrino_interaction: a numu with a muon
# among its first daughters is numu_CC, a nue with an electron is nue_CC, neither
# is NC. Cosmic true clusters have no channel and drop out, as they do for the
# volume split.
CHANNEL_DIRNAME = "By_Interaction_Channel"
CHANNELS = [
    {'key': 'numu_CC', 'dirname': 'numu_CC', 'label': 'true numu CC interactions'},
    {'key': 'nue_CC',  'dirname': 'nue_CC',  'label': 'true nue CC interactions'},
    {'key': 'NC',      'dirname': 'NC',      'label': 'true NC interactions'},
]

IN_OUT_README = """\
This directory holds the evaluation restricted to {label}.

Only the TRUE side is filtered. Every reco cluster that passed the beam window cut
is still present and every efficiency/purity value here is the one computed against
that full reco population -- a purity in this directory still measures the cosmic
contamination of the reco cluster its true neutrino matched.

Not here, by design:
  - cosmic TRUE clusters (they are neither in- nor out-of-volume)
  - unmatched RECO clusters (purity rows with true_cluster_id=8888): a reco cluster
    that touched no true cluster belongs to neither volume
  - imaging_details/, clustering_details/, reco_cluster_info.txt: flashes and reco
    clusters carry no truth label, so they are drawn once only

All of those live in ../All_True_Clusters/, which is the complete evaluation
(cosmic and neutrino true clusters, both volumes) -- the same output this notebook
produced before the split.
"""


def _with_population(text, population_label):
    """Append the population label to a title-only string (level_name / file_name)."""
    if not population_label:
        return text
    return f"{text} ({population_label})" if text else population_label


def channel_map_for_population(channel_map, volume_map, population_key):
    """
    The channel map restricted to one vertex-volume population, so the two splits
    can be composed: filtering with this gives the numu CC / nue CC / NC slices OF
    the in-volume neutrinos, or of the out-of-volume ones.

    population_key='all' returns the channel map untouched (both volumes), which
    is what the All root wants. Interactions with no volume flag drop out of the
    'in'/'out' maps, exactly as they do from the volume roots themselves.
    """
    if population_key == 'all':
        return channel_map
    return restrict_label_map(channel_map, volume_map, population_key)


def filter_population_records(label_map, key, efficiency_results=None, purity_results=None,
                              metadata_list=None, pair_metadata_list=None,
                              cluster_type_records=None, vertex_records=None,
                              matched_true_reco_clusters=None):
    """
    One population's slice of every record list a render function needs, keyed by
    (event, true cluster id) through metadata.filter_records_by_label.

    Serves both splits: pass a volume map ('all'/'in'/'out') or a channel map
    ('numu_CC'/'nue_CC'/'NC') -- the filter only compares the map's value to `key`,
    so the two are the same operation on different labels. key='all' passes every
    list through untouched.

    Returns a dict with the same names as the arguments, so it can be splatted
    into render_event_outputs / render_summary_outputs.
    """
    def _f(records, id_key='true_cluster_id'):
        return filter_records_by_label(records or [], label_map, key, id_key=id_key)

    return {
        'efficiency_results':         _f(efficiency_results),
        'purity_results':             _f(purity_results),
        'metadata_list':              _f(metadata_list),
        'pair_metadata_list':         _f(pair_metadata_list),
        'matched_true_reco_clusters': _f(matched_true_reco_clusters),
        # These two carry the true cluster id under 'cluster_id', not
        # 'true_cluster_id' -- see build_true_cluster_type_records /
        # build_neutrino_vertex_records.
        'cluster_type_records':       _f(cluster_type_records, id_key='cluster_id'),
        'vertex_records':             _f(vertex_records, id_key='cluster_id'),
    }


def render_event_outputs(event_dir, evt, apa, file_name,
                         clusters_true, clusters_reco,
                         efficiency_results, purity_results,
                         metadata_list, pair_metadata_list,
                         matched_true_reco_clusters, cluster_category_results,
                         cluster_type_records, vertex_records,
                         clusters_true_full=None, clusters_reco_full=None,
                         reco_beam_window_record=None,
                         population_label=None, draw=True):
    """
    Everything this notebook writes for ONE event and ONE population, into
    event_dir. Called once per population per event; with the unfiltered records
    it reproduces the event output this notebook produced before the split, file
    for file.

    Parameters:
    - event_dir: the event's output directory (created here)
    - evt, apa, file_name: identification, as everywhere else
    - clusters_true: true clusters of THIS population (the spatial views' left
      panel); clusters_reco: ALL beam-window reco clusters (right panel)
    - efficiency_results, purity_results, metadata_list, pair_metadata_list,
      matched_true_reco_clusters, cluster_category_results, cluster_type_records,
      vertex_records: this population's slice of the event's records
    - clusters_true_full / clusters_reco_full: the unfiltered dicts, for
      DrawTrueClusterWithMatchedReco -- it indexes into them by cluster id and
      must be able to reach every reco cluster a true cluster matched, including
      the ones outside this population. Default to the passed-in dicts.
    - reco_beam_window_record: reco-side record; pass it in the All root only
    - population_label: appended to titles; None in the All root
    - draw: False writes the text tables and skips every plot
    """
    if clusters_true_full is None:
        clusters_true_full = clusters_true
    if clusters_reco_full is None:
        clusters_reco_full = clusters_reco

    event_dir.mkdir(parents=True, exist_ok=True)

    efficiency_dir = event_dir / "efficiency"
    purity_dir     = event_dir / "purity"
    efficiency_dir.mkdir(parents=True, exist_ok=True)
    purity_dir.mkdir(parents=True, exist_ok=True)

    # Sub-directories for 2D/1D efficiency-vs-true-energy plots, split by computation
    # style (see the main loop's header note): "imaginglevel" (every true cluster,
    # summed efficiency across all reco matches, no purity), "clusteringlevel" (1-to-1
    # best match + unmatched true clusters at efficiency=0, with purity),
    # "clusteringlevel_true_reco_pairs_only" (1-to-1 matched pairs only, no unmatched).
    efficiency_2d1d_imaging_dir               = efficiency_dir / "efficiency_2d_1d_imaginglevel"
    efficiency_2d1d_clustering_dir            = efficiency_dir / "efficiency_2d_1d_clusteringlevel"
    efficiency_2d1d_clustering_pairs_only_dir = efficiency_dir / "efficiency_2d_1d_clusteringlevel_true_reco_pairs_only"
    efficiency_2d1d_imaging_dir              .mkdir(parents=True, exist_ok=True)
    efficiency_2d1d_clustering_dir           .mkdir(parents=True, exist_ok=True)
    efficiency_2d1d_clustering_pairs_only_dir.mkdir(parents=True, exist_ok=True)

    # One including unmatched true clusters (drawn in a dedicated "no match" box),
    # one excluding them
    eff_vs_purity_incl_dir = event_dir / "true_reco_matched_pair_efficiency_purity_including_unmatched_true_clusters"
    eff_vs_purity_excl_dir = event_dir / "true_reco_matched_pair_efficiency_purity_excluding_unmatched_true_clusters"
    eff_vs_purity_incl_dir.mkdir(parents=True, exist_ok=True)
    eff_vs_purity_excl_dir.mkdir(parents=True, exist_ok=True)

    title_file  = _with_population(file_name, population_label)
    title_level = _with_population(f"Event {evt}", population_label)

    if draw:
        # neutrino_records: the banner above each XZ/YZ/XY canvas naming every true
        # neutrino drawn on it -- cluster id, interaction channel, vertex volume --
        # so two neutrinos in one event can be told apart by the same ids the
        # legends and the .txt tables use.
        DrawTrueRecoClustersXZ(clusters_true, clusters_reco, evt, apa, event_dir, title_file,
                               neutrino_records=vertex_records)
        DrawTrueRecoClustersYZ(clusters_true, clusters_reco, evt, apa, event_dir, title_file,
                               neutrino_records=vertex_records)
        DrawTrueRecoClustersXY(clusters_true, clusters_reco, evt, apa, event_dir, title_file,
                               neutrino_records=vertex_records)
        # Same three views again with the left panel restricted to the true NEUTRINO
        # clusters (right panel unchanged: all selected/beam-window reco clusters), so the
        # neutrino can be found without the cosmic tracks covering it:
        # neutrino_clusters_reco_true_event*_apa_Combined_{XZ,YZ,XY}.png
        DrawNeutrinoRecoClusters(clusters_true, clusters_reco, evt, apa, event_dir, title_file,
                                 neutrino_records=vertex_records)
        DrawLabelsAggregated(cluster_type_records, event_dir, title_level, f"event_{evt}", apa,
                             file_name=title_file, vertex_records=vertex_records)
        DrawLabelsByNuIdx(cluster_type_records, event_dir, title_level, f"event_{evt}", apa,
                          file_name=title_file)
        DrawNeutrinoVertices(vertex_records, event_dir, title_level, f"event_{evt}", apa,
                             file_name=title_file,
                             x_min=x_min, x_max=x_max, y_min=y_min, y_max=y_max, z_min=z_min, z_max=z_max)
        DrawNeutrinoVolumeCategory(vertex_records, event_dir, title_level, f"event_{evt}", apa,
                                   file_name=title_file)
        DrawNeutrinoFlavor(vertex_records, event_dir, title_level, f"event_{evt}", apa,
                           file_name=title_file)
        DrawNeutrinoBreakdown(vertex_records, event_dir, title_level, f"event_{evt}", apa,
                              file_name=title_file, energy_threshold=min_cluster_energy)
        plot_efficiency_heatmap(efficiency_results, evt, apa, efficiency_dir, title_file)
        plot_purity_heatmap(purity_results, evt, apa, purity_dir, title_file)

        DrawTrueRecoMatchMultiplicity(metadata_list, event_dir, apa, title_level, f"event_{evt}",
                                      file_name=title_file)
        # Full cluster dicts on purpose -- a true cluster in this population can be
        # matched to reco clusters the population filter never touched.
        for matched_info in matched_true_reco_clusters:
            DrawTrueClusterWithMatchedReco(matched_info, clusters_true_full, clusters_reco_full,
                                           efficiency_dir, evt, apa, title_file,
                                           neutrino_records=vertex_records)

        DrawEfficiencyVsTrueEnergyPerEvent(efficiency_results, efficiency_2d1d_imaging_dir, evt, apa,
                                           title_file, cluster_category_results=cluster_category_results)
        DrawClusterEfficiencyVsTrueEnergyPerEvent(pair_metadata_list, efficiency_2d1d_clustering_dir, evt, apa,
                                                  title_file, all_true_metadata_list=metadata_list)
        DrawEfficiencyVsTrueEnergy_MatchedPairs_PerEvent(pair_metadata_list, efficiency_2d1d_clustering_pairs_only_dir,
                                                        evt, apa, title_file)
        DrawPurityVsRecoChargePerEvent(pair_metadata_list, purity_dir, evt, apa, title_file)
        # filename_level: the plain level, so the population label stays in the
        # title and out of the filename -- same plot, same name, every root.
        DrawEfficiencyVsPurity_MatchedPairs(pair_metadata_list, eff_vs_purity_incl_dir, title_level, apa,
                                            title_file, all_true_metadata_list=metadata_list,
                                            filename_level=f"Event {evt}")
        DrawEfficiencyVsPurity_MatchedPairs(pair_metadata_list, eff_vs_purity_excl_dir, title_level, apa,
                                            title_file, all_true_metadata_list=None,
                                            filename_level=f"Event {evt}")
        plt.close('all')

    # ------------------------------------------------------------------
    # TEXT FILES FOR CONFIRMATION: the exact underlying efficiency/purity data
    # behind each directory's plots, so cluster-by-cluster values can be checked
    # directly instead of only being read off a plot. These are the SAME writers
    # called again at file and job level, so an event-level file is that event's
    # slice of the aggregated one, byte for byte, and the levels can never drift
    # apart in format. That is why they carry an 'event' column even here, where
    # it is constant.
    # ------------------------------------------------------------------
    write_efficiency_info(metadata_list, efficiency_2d1d_imaging_dir)
    write_pair_efficiency_info(pair_metadata_list, efficiency_2d1d_clustering_dir,
                               all_true_metadata_list=metadata_list)
    write_pair_efficiency_info(pair_metadata_list, efficiency_2d1d_clustering_pairs_only_dir)
    write_true_cluster_info(cluster_type_records, event_dir)
    # Reco-side: written in the All root only (the caller passes None elsewhere).
    if reco_beam_window_record is not None:
        write_reco_cluster_info([reco_beam_window_record], event_dir)
    write_neutrino_vertex_info(vertex_records, event_dir)
    write_removed_neutrino_info(vertex_records, event_dir)
    # raw EvaluatePurity output, one row per reco cluster
    write_purity_info(purity_results, purity_dir)


def render_summary_outputs(summary_dir, level,
                           efficiency_results, purity_results,
                           metadata_list, pair_metadata_list,
                           cluster_type_records, vertex_records,
                           apa="Combined", file_name=None,
                           population_label=None, draw=True,
                           include_cosmic_plots=True):
    """
    Everything this notebook writes for ONE aggregation level and ONE population,
    into summary_dir. Same set as render_event_outputs, drawn with the PerFile or
    PerJob variant of each drawer.

    Parameters:
    - summary_dir: file_summary/ or job_summary/ (created here)
    - level: 'file' or 'job' -- selects the drawer variant, the title and the
      filename prefix
    - efficiency_results ... vertex_records: this population's slice of the
      level's accumulated records
    - file_name: the input file at file level; None at job level (a job spans
      every file, so there is no one file to name)
    - population_label: appended to titles; None in the All root
    - draw: False writes the text tables and skips every plot
    - include_cosmic_plots: False in the In/Out roots, where the population is
      neutrinos only and the cosmic-category bar would be empty
    """
    summary_dir.mkdir(parents=True, exist_ok=True)

    level_name       = "File Level" if level == 'file' else "Job Level"
    filename_prefix  = level
    title_level      = _with_population(level_name, population_label)
    title_file       = _with_population(file_name, population_label)

    # Deliberately ahead of the efficiency/purity block and outside its guard:
    # these describe the true neutrino population, which exists whether or not
    # this level produced any efficiency/purity record. (true_cluster_info.txt is
    # a job-level file only, as before -- one row per event, so a per-file copy
    # would only ever repeat rows the job-level file already has.)
    write_neutrino_vertex_info(vertex_records, summary_dir)
    write_removed_neutrino_info(vertex_records, summary_dir)
    if level == 'job':
        write_true_cluster_info(cluster_type_records, summary_dir)

    # The neutrino-selection drawers, likewise outside the efficiency guard: they
    # describe the true population, not the reconstruction of it. Every one of
    # them returns early on empty input.
    if draw:
        DrawLabelsAggregated(cluster_type_records, summary_dir, title_level, filename_prefix, apa,
                             file_name=title_file, vertex_records=vertex_records)
        DrawLabelsByNuIdx(cluster_type_records, summary_dir, title_level, filename_prefix, apa,
                          file_name=title_file)
        DrawNeutrinoVertices(vertex_records, summary_dir, title_level, filename_prefix, apa,
                             file_name=title_file,
                             x_min=x_min, x_max=x_max, y_min=y_min, y_max=y_max, z_min=z_min, z_max=z_max)
        DrawNeutrinoVolumeCategory(vertex_records, summary_dir, title_level, filename_prefix, apa,
                                   file_name=title_file)
        DrawNeutrinoFlavor(vertex_records, summary_dir, title_level, filename_prefix, apa,
                           file_name=title_file)
        DrawNeutrinoBreakdown(vertex_records, summary_dir, title_level, filename_prefix, apa,
                              file_name=title_file, energy_threshold=min_cluster_energy)

    # 'or', not 'and': a file/job whose every event lost all its reco clusters to
    # the beam-window cut has no purity records at all, but its true clusters are
    # still real reconstruction failures that belong in the efficiency plots and
    # tables. Every drawer below tolerates an empty pair/purity list.
    if not (efficiency_results or purity_results):
        return

    efficiency_2d1d_imaging_dir               = summary_dir / "efficiency" / "efficiency_2d_1d_imaginglevel"
    efficiency_2d1d_clustering_dir            = summary_dir / "efficiency" / "efficiency_2d_1d_clusteringlevel"
    efficiency_2d1d_clustering_pairs_only_dir = summary_dir / "efficiency" / "efficiency_2d_1d_clusteringlevel_true_reco_pairs_only"
    efficiency_2d1d_imaging_dir              .mkdir(parents=True, exist_ok=True)
    efficiency_2d1d_clustering_dir           .mkdir(parents=True, exist_ok=True)
    efficiency_2d1d_clustering_pairs_only_dir.mkdir(parents=True, exist_ok=True)

    purity_summary_dir = summary_dir / "purity"
    purity_summary_dir.mkdir(parents=True, exist_ok=True)

    eff_vs_purity_incl_dir = summary_dir / "true_reco_matched_pair_efficiency_purity_including_unmatched_true_clusters"
    eff_vs_purity_excl_dir = summary_dir / "true_reco_matched_pair_efficiency_purity_excluding_unmatched_true_clusters"
    eff_vs_purity_incl_dir.mkdir(parents=True, exist_ok=True)
    eff_vs_purity_excl_dir.mkdir(parents=True, exist_ok=True)

    # Written regardless of `draw` -- these are the numbers behind the plots,
    # worth having even on a run with drawing turned off.
    write_efficiency_info(metadata_list, efficiency_2d1d_imaging_dir)
    write_pair_efficiency_info(pair_metadata_list, efficiency_2d1d_clustering_dir,
                               all_true_metadata_list=metadata_list)
    write_pair_efficiency_info(pair_metadata_list, efficiency_2d1d_clustering_pairs_only_dir)
    write_purity_info(purity_results, purity_summary_dir)

    if not draw:
        return

    if level == 'file':
        DrawEfficiencyVsTrueEnergyPerFile(efficiency_results, efficiency_2d1d_imaging_dir, apa,
                                          title_file, file_metadata_list=metadata_list)
        DrawClusterEfficiencyVsTrueEnergyPerFile(pair_metadata_list, efficiency_2d1d_clustering_dir, apa,
                                                 title_file, all_true_metadata_list=metadata_list)
        DrawEfficiencyVsTrueEnergy_MatchedPairs_PerFile(pair_metadata_list,
                                                        efficiency_2d1d_clustering_pairs_only_dir, apa, title_file)
        DrawPurityVsRecoChargePerFile(pair_metadata_list, purity_summary_dir, apa, title_file)
    else:
        # The PerJob drawers take neither file_name nor level_name, so their
        # titles carry no population label -- the directory name and summary.txt
        # identify them.
        DrawEfficiencyVsTrueEnergyPerJob(efficiency_results, efficiency_2d1d_imaging_dir, apa,
                                         job_metadata_list=metadata_list)
        DrawClusterEfficiencyVsTrueEnergyPerJob(pair_metadata_list, efficiency_2d1d_clustering_dir, apa,
                                                all_true_metadata_list=metadata_list)
        DrawEfficiencyVsTrueEnergy_MatchedPairs_PerJob(pair_metadata_list,
                                                       efficiency_2d1d_clustering_pairs_only_dir, apa)
        DrawPurityVsRecoChargePerJob(pair_metadata_list, purity_summary_dir, apa)

    DrawEfficiencyVsPurity_MatchedPairs(pair_metadata_list, eff_vs_purity_incl_dir, title_level, apa,
                                        title_file, all_true_metadata_list=metadata_list,
                                        filename_level=level_name)
    DrawEfficiencyVsPurity_MatchedPairs(pair_metadata_list, eff_vs_purity_excl_dir, title_level, apa,
                                        title_file, all_true_metadata_list=None,
                                        filename_level=level_name)
    if include_cosmic_plots:
        # Cosmic track-geometry categorisation -- empty by construction in the
        # In/Out roots, which hold neutrinos only.
        _draw_cosmic_category_bar(metadata_list, summary_dir, apa, title_level, filename_prefix,
                                  file_name=title_file)
    DrawTrueRecoMatchMultiplicity(metadata_list, summary_dir, apa, title_level, filename_prefix,
                                  file_name=title_file)
    plt.close('all')


print("Population roots:", ", ".join(p['dirname'] for p in POPULATIONS))

Population roots: All_True_Clusters, In_Volume_True_Neutrinos, Out_Volume_True_Neutrinos


: 

In [ ]:
# ============================================================================
# MAIN PROCESSING LOOP -- combined-APA charge-light matching evaluation
# ============================================================================
# CLUSTERING-LEVEL evaluation (after charge-light-matching, CLM): selections /
# cluster_category / efficiency / purity / 1-to-1 & 1-to-many matching /
# metadata / drawing are all computed on sed-smear_readout (true, no SCE
# correction) vs clustering-global (reco, post-CLM) -- NOT img-global/sed-sce
# (imaging-level, pre-CLM), which are still read (for the flash-bridging
# logic below) but no longer feed the efficiency/purity/matching pipeline.
# reassign_cluster_ID_true_charge_light IS used (true clusters grouped under
# 99990+nu_idx -- 99991, 99992, ... one per neutrino interaction -- instead
# of a single shared 9999, so multiple neutrino interactions in the same
# event are no longer merged into one true cluster; avg-X for cosmic,
# unchanged). See selections.py for why this is a separate function from the
# legacy reassign_cluster_ID_true (still 9999-merging, still used by other
# pipelines). reassign_cluster_ID_reco IS ALSO used (reco clusters relabeled
# by avg-X) -- but only for the efficiency/purity/matching pipeline's
# clusters_reco; the earlier clusters_all_clu (flash-time bridging /
# beam-window highlighting) stays keyed by clustering-global's ORIGINAL
# cluster_id, since that's the namespace build_img_cluster_flash_metadata's
# records reference.
#
# IMPORTANT: clustering-global points are grouped by REAL_CLUSTER_ID, not
# cluster_id -- clustering-global's own 'cluster_id' is a COARSER grouping
# that can merge multiple physically distinct tracks together (confirmed
# against real data via BEE display comparison: a single cluster_id spanning
# two disjoint Y ranges that split cleanly into two real_cluster_id values,
# each matching a different true cluster). 'real_cluster_id' is the
# physically correct per-track ID. (img-global does NOT have this issue --
# cluster_id == real_cluster_id everywhere there -- so img-global-side
# grouping is unaffected and still uses cluster_id.)
#
# q_true is now taken from sed-smear's 'nu_idx' field when available (0=cosmic,
# 1/2/...=which neutrino interaction), not just a binary 0/1 flag -- see
# build_true_points_charge_light's nu_idx= parameter. Combined with
# reassign_cluster_ID_true_charge_light's 99990+nu_idx scheme above, each
# neutrino interaction is now its own true cluster, so DrawLabelsAggregated's
# "Neutrino" cluster count and the sum of DrawLabelsByNuIdx's per-nu_idx bars
# now agree (previously they diverged, since all neutrino interactions were
# merged under one cluster_id=9999 and DrawLabelsByNuIdx counted nu_idx
# values found inside that single merged cluster). NOTE: apply_energy_cutoff
# / apply_min_true_points_cutoff run AFTER reassignment, so with this scheme
# each neutrino interaction's energy/point-count is now cut independently,
# rather than pooled together as before. The pre-existing
# DrawLabels/DrawLabelPerFile calls stay as-is (unmodified, additive-only)
# but their strict q_true==1 point-level check still only recognizes
# nu_idx=1 as neutrino, undercounting nu_idx=2/3 clusters -- unaffected by
# this ID scheme change, since that check never looked at cluster_id. At job
# level, DrawLabelPerJob itself is no longer called (its neutrino-vs-cosmic
# bar, true_clusters_by_type_job_*events_Combined.png, isn't wanted) -- only
# its cosmic-category-breakdown half is still drawn, via
# _draw_cosmic_category_bar directly (same helper DrawLabelPerJob itself
# calls internally), so cosmic_clusters_by_category_job_*events_Combined.png
# is unaffected.
#
# Note on the "imaginglevel"/"clusteringlevel" directory names below: since
# EVERYTHING here is post-CLM data now, these names no longer distinguish
# data source (that's what "afterCLM" used to flag, now dropped as redundant)
# -- they instead distinguish COMPUTATION STYLE, echoing the original
# pipeline's naming: "imaginglevel" = direct per-true-cluster efficiency
# (DrawEfficiencyVsTrueEnergyPerEvent, summed across all reco matches, no
# purity); "clusteringlevel" = 1-to-1 best-match pairing
# (DrawClusterEfficiencyVsTrueEnergyPerEvent / MatchedPairs variant, with
# purity attached).
#
# BEAM_WINDOW_MIN_US / BEAM_WINDOW_MAX_US imported from DrawRecoTrueFlashes.
#
# THREE OUTPUT ROOTS. The evaluation below runs ONCE per event, against the full
# true and reco populations; its finished records are then rendered three times by
# render_event_outputs / render_summary_outputs (previous cell) into
#   All_True_Clusters/          -- everything, cosmic and neutrino, both volumes
#   In_Volume_True_Neutrinos/   -- true neutrinos whose mc.json vertex is inside
#                                  the wire-readout sensitive box
#   Out_Volume_True_Neutrinos/  -- true neutrinos whose vertex is outside it
# Only the TRUE side is filtered; the reco side is never cut, so an efficiency or
# purity value is identical in every root it appears in. See the previous cell's
# header for what that means for cosmic true clusters and unmatched reco clusters.

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = PLOTBASEDIR / f"combined_apa_{timestamp}"
output_dir.mkdir(parents=True, exist_ok=True)

# THREE OUTPUT ROOTS, one per true-cluster population (see the POPULATIONS cell
# above): the complete evaluation, and the same evaluation restricted to true
# neutrinos whose interaction vertex is inside / outside the wire-readout
# sensitive box. The evaluation runs ONCE and is rendered three times from
# filtered copies of its records -- nothing is recomputed, and only the TRUE side
# is ever filtered (every beam-window reco cluster stays in play in all three).
population_dirs = {}
for _population in POPULATIONS:
    _root = output_dir / _population['dirname']
    _root.mkdir(parents=True, exist_ok=True)
    population_dirs[_population['key']] = _root
    if _population['label']:
        (_root / "README.txt").write_text(IN_OUT_README.format(label=_population['label']))

# The All root is where the reco-side-only diagnostics go: optical flashes,
# img-global / clustering-global cluster drawings and reco_cluster_info.txt carry
# no truth label, so they are drawn once rather than copied three times.
all_output_dir = population_dirs['all']

print(f"\n{'='*70}")
print(f"Output directory: {output_dir}")
for _population in POPULATIONS:
    print(f"  {_population['dirname']}/"
          f"{'   (reco-side diagnostics live here)' if _population['label'] is None else ''}")
print(f"{'='*70}\n")

job_efficiency_results          = []
job_purity_results              = []
job_metadata_list               = []      # per-true-cluster metadata (add_metadata_true_clusters)
job_pair_metadata_list          = []      # per 1-to-1 true-reco pair metadata (add_metadata_true_reco_pair_cluster)
job_flash_metadata_list         = []      # per-cluster flash records (build_cluster_flash_metadata)
job_img_cluster_flash_records   = []      # per-clustering-cluster flash records (build_img_cluster_flash_metadata)
job_cluster_type_records        = []     # per-true-cluster is_neutrino records (build_true_cluster_type_records)
job_vertex_records              = []      # per true neutrino interaction (build_neutrino_vertex_records)
job_reco_beam_window_records    = []      # per-event count of clustering-global clusters with a beam-window flash
total_events_processed          = 0
total_files_processed           = 0

# Wall-clock seconds spent RENDERING each population root (event + file + job
# level), for the top-level summary.txt. This is drawing and writing time only --
# the evaluation itself runs once, outside any population loop, and is charged to
# none of them. The three do not sum to the job runtime for exactly that reason.
population_render_seconds = {population['key']: 0.0 for population in POPULATIONS}

for file_idx, input_dir in enumerate(input_directories):
    input_file_name = input_dir.name

    # SELECTIVE FILTERING: Skip files that don't match target_file
    if target_file is not None and input_file_name != target_file:
        print(f"Skipping {input_file_name} (target: {target_file})")
        continue

    # SELECTIVE FILTERING: Skip files outside target_file_range (inclusive both
    # ends, matched on the directory name's trailing index -- see
    # file_index_from_name). Applied on top of target_file, not instead of it.
    if target_file_range is not None:
        file_idx = file_index_from_name(input_file_name)
        range_low, range_high = target_file_range
        if file_idx is None or not (range_low <= file_idx <= range_high):
            print(f"Skipping {input_file_name} (target range: file{range_low}..file{range_high})")
            continue

    print(f"\n{'='*70}")
    print(f"FILE {file_idx+1}/{len(input_directories)}: {input_dir}")
    print(f"{'='*70}")

    # Per-population file directories. The All root's is created up front (the
    # reco-side diagnostics and every event directory go there); the In/Out ones
    # are created lazily, only for events/files that actually have a neutrino of
    # that kind, so the tree does not fill with empty directories.
    file_output_dirs = {key: root / input_file_name for key, root in population_dirs.items()}
    file_output_dir  = file_output_dirs['all']
    file_output_dir.mkdir(parents=True, exist_ok=True)

    # Containers for file-level aggregation
    file_efficiency_results         = []
    file_purity_results             = []
    file_metadata_list              = []
    file_pair_metadata_list         = []
    file_flash_metadata_list        = []
    file_img_cluster_flash_records  = []
    file_cluster_type_records       = []
    file_reco_beam_window_records   = []
    file_vertex_records             = []      # per true neutrino interaction (build_neutrino_vertex_records)

    events_list = detect_events_in_directory(input_dir)
    if not events_list:
        print(f"No events found in {input_dir}, skipping...")
        continue

    event_low = min(events_list)
    if num_events_to_process is None:
        event_high = max(events_list) + 1  # Process all events
    else:
        event_high = event_low + num_events_to_process

    print(f"Processing events {event_low} to {event_high-1}\n")
    total_files_processed += 1


    # Start of event loop
    for evt in range(event_low, event_high):
        # SELECTIVE FILTERING: Skip events that don't match target_event
        if target_event is not None and evt != target_event:
            continue

        # SELECTIVE FILTERING: Skip events outside target_event_range (inclusive
        # both ends). Applied on top of target_event, not instead of it.
        if target_event_range is not None:
            event_range_low, event_range_high = target_event_range
            if not (event_range_low <= evt <= event_range_high):
                continue

        result = read_charge_light_files_for_event(input_dir, evt)
        if result is None:
            print(f"  Event {evt}: could not read data, skipping")
            continue

        event_key = f"{input_file_name}_{evt}"
        # The event directory of the ALL root. The per-population copies are made
        # by render_event_outputs itself, further down; the efficiency/purity/
        # matched-pair subdirectories that used to be created here are created
        # there too, once per population.
        event_output_dir = file_output_dir / f"event_{evt:03d}"
        event_output_dir.mkdir(parents=True, exist_ok=True)
        PLOTDIR_EVT = event_output_dir

        # Sub-directories for imaging-level flash/cluster diagnostics
        imaging_details_flashes_dir  = event_output_dir / "imaging_details" / "flashes"
        imaging_details_clusters_dir = event_output_dir / "imaging_details" / "clusters"
        imaging_details_flashes_dir .mkdir(parents=True, exist_ok=True)
        imaging_details_clusters_dir.mkdir(parents=True, exist_ok=True)

        # Sub-directories for clustering-level (post charge-light-matching) diagnostics
        clustering_details_flashes_dir  = event_output_dir / "clustering_details" / "flashes"
        clustering_details_clusters_dir = event_output_dir / "clustering_details" / "clusters"
        clustering_details_flashes_dir .mkdir(parents=True, exist_ok=True)
        clustering_details_clusters_dir.mkdir(parents=True, exist_ok=True)

        x_reco, y_reco, z_reco, id_reco, q_reco, real_id_reco = result['reco']
        # sed-smear_readout (NOT sed-sce) -- clustering-level (post-CLM) truth, paired
        # with clustering-global's reco below. sed-sce (imaging-level, pre-CLM truth,
        # result['true']) is still returned by read_charge_light_files_for_event but
        # no longer feeds this pipeline.
        x_true, y_true, z_true, id_true, q_true, real_id_true, e_true, nu_idx_true = result['true_clustering']
        mc_tree = result['mc']
        op_data = result['op']

        # ------------------------------------------------------------------
        # FLASH METADATA (op.json)
        # ------------------------------------------------------------------
        event_flash_metadata_list = build_cluster_flash_metadata(op_data, input_file_name, evt, "Combined", event_key)
        if b_draw_event_level_plots:
            draw_flashes(event_flash_metadata_list, imaging_details_flashes_dir, "Combined", f"Event {evt}", f"event_{evt}", file_name=input_file_name, write_text_table=True)  

        # ------------------------------------------------------------------
        # IMG-GLOBAL CLUSTERS (event level only): grouped by the RAW
        # img-global cluster_id (point-wise, matched by array index) --
        # NOT reassigned via reassign_cluster_ID_reco, since that's the
        # namespace op_cluster_ids uses to reference clusters. (img-global's
        # cluster_id == real_cluster_id everywhere, so no distinction here.)
        # ------------------------------------------------------------------
        predicted_points_raw = np.column_stack((x_reco, y_reco, z_reco, id_reco, q_reco))
        clusters_all_raw = GroupClustersByID(predicted_points_raw)

        flash_matched_ids = {float(r['reco_cluster_id']) for r in event_flash_metadata_list}
        beam_window_ids   = {float(r['reco_cluster_id']) for r in event_flash_metadata_list
                              if BEAM_WINDOW_MIN_US <= r['flash_time'] <= BEAM_WINDOW_MAX_US}

        clusters_with_flash      = {cid: pts for cid, pts in clusters_all_raw.items() if cid in flash_matched_ids}
        clusters_in_beam_window  = {cid: pts for cid, pts in clusters_all_raw.items() if cid in beam_window_ids}

        if b_draw_event_level_plots:
            draw_img_global_clusters(clusters_all_raw, clusters_with_flash, clusters_in_beam_window,
                                     evt, "Combined", imaging_details_clusters_dir, file_name=input_file_name)

        # ------------------------------------------------------------------
        # CLUSTERING-GLOBAL <-> IMG-GLOBAL FLASH BRIDGE (event level for the
        # spatial/bar plots; the num-clusters-vs-flash-time plot also
        # aggregates to file/job level below). clustering-global's cluster_id
        # is a different numbering scheme than img-global's -- the bridge is
        # by point-level charge ('q') value, see build_img_cluster_flash_metadata.
        # clusters_all_clu here is grouped by clustering-global's
        # REAL_CLUSTER_ID (NOT cluster_id, which merges distinct tracks --
        # see header note; also NOT reassign_cluster_ID_reco'd) since
        # real_cluster_id is the namespace build_img_cluster_flash_metadata's
        # records now reference.
        # ------------------------------------------------------------------
        event_img_cluster_flash_records = build_img_cluster_flash_metadata(
            result['reco'], result['clustering'], event_flash_metadata_list,
            input_file_name, evt, "Combined", event_key)

        if b_draw_event_level_plots:
            draw_clustering_flashes(event_img_cluster_flash_records, clustering_details_flashes_dir, "Combined",
                                     f"Event {evt}", f"event_{evt}", file_name=input_file_name, write_text_table=True)
            draw_cluster_flash_time_bar(event_img_cluster_flash_records, evt, "Combined",
                                         clustering_details_flashes_dir, file_name=input_file_name)

        x_clu, y_clu, z_clu, id_clu, q_clu, real_id_clu = result['clustering']
        predicted_points_clu = np.column_stack((x_clu, y_clu, z_clu, real_id_clu, q_clu))
        clusters_all_clu = GroupClustersByID(predicted_points_clu)

        clu_beam_window_ids = {float(r['clustering_cluster_id']) for r in event_img_cluster_flash_records
                                if BEAM_WINDOW_MIN_US <= r['flash_time'] <= BEAM_WINDOW_MAX_US}
        clusters_clu_in_beam_window = {cid: pts for cid, pts in clusters_all_clu.items() if cid in clu_beam_window_ids}

        # Per-event reco-side beam-window record (job level: reco_cluster_info.txt) --
        # counts DISTINCT clustering-global clusters with a beam-window-matched flash,
        # independent of any true-cluster info: a proxy for multiple neutrino-like
        # activity in the beam spill from the RECO side (unlike true_cluster_info.txt's
        # num_neutrinos, which is ground truth via nu_idx).
        event_reco_beam_window_record = {
            'file_name': input_file_name,
            'event': event_key,
            'event_num': evt,
            'num_clusters_in_beam_window': len(clusters_clu_in_beam_window),
            'cluster_ids': sorted(clusters_clu_in_beam_window.keys()),
        }


        if b_draw_event_level_plots:
            draw_clustering_global_clusters(clusters_all_clu, clusters_clu_in_beam_window,
                                             evt, "Combined", clustering_details_clusters_dir, file_name=input_file_name)

        # ------------------------------------------------------------------
        # TRUE POINTS: adapt sed-smear_readout into the standard 7-column
        # shape (energy column is sed-smear's per-point 'e' field, genuine
        # MeV; q_true column is sed-smear's own 'nu_idx' field when available
        # -- 0=cosmic, 1/2/...=which neutrino interaction), reassign IDs
        # (99990+nu_idx for neutrino -- one cluster per interaction -- avg-X
        # for cosmic), then apply the same selection cuts as the existing
        # pipeline.
        # ------------------------------------------------------------------
        # real_id_true (real_cluster_id), NOT id_true (cluster_id): same reason as the
        # reco side -- clustering-global's cluster_id is a coarser grouping that can
        # merge physically distinct tracks together, real_cluster_id is the physically
        # correct per-track ID. The two arrays are identical in every sed-smear file in
        # the current tree, so this changes no present result -- it makes the true side
        # robust the way the reco side already is.
        true_points = build_true_points_charge_light(x_true, y_true, z_true, real_id_true, q_true, energy=e_true, nu_idx=nu_idx_true)
        true_points = reassign_cluster_ID_true_charge_light(true_points)

        # Snapshot BEFORE the cuts: build_neutrino_vertex_records uses it to say
        # what a removed neutrino actually deposited, and hence which cut removed
        # it (removed_true_neutrino_info.txt). Grouping only, no filtering.
        clusters_true_precut = GroupClustersByID(true_points)

        if Apply_energy_cutoff:
            true_points = apply_energy_cutoff(true_points, min_cluster_energy)
        if Apply_min_true_points_cutoff:
            true_points = apply_min_true_points_cutoff(true_points, min_true_points_cutoff)
        if Apply_wire_readout_sensitive_xz_plane_cut:
            true_points = apply_wire_readout_sensitive_yz_plane_cut_true(true_points, x_min, x_max, y_min, y_max, z_min, z_max)
        if Apply_deadarea_cut:
            true_points = apply_deadarea_cut_true_charge_light(true_points, output_dir=PLOTDIR_EVT, event=evt, file_name=input_file_name)

        if len(true_points) == 0:
            print(f"  Event {evt}: no true points remain after cuts, skipping")
            continue

        clusters_true = GroupClustersByID(true_points)

        # ------------------------------------------------------------------
        # RECO POINTS: clustering-global (post-CLM), grouped by
        # REAL_CLUSTER_ID (not cluster_id -- see header note: cluster_id can
        # merge physically distinct tracks together, real_cluster_id is the
        # correct per-track ID, confirmed against BEE display). Cluster IDs
        # ARE further reassigned via reassign_cluster_ID_reco (relabeled by
        # avg-X, same convention as the true side) -- applied AFTER the
        # cuts, matching the original pipeline's ordering.
        # ------------------------------------------------------------------
        predicted_points = np.column_stack((x_clu, y_clu, z_clu, real_id_clu, q_clu))

        # BEAM-WINDOW CUT -- the only thing separating this notebook from
        # Evaluation_ChargeLightMatching_BeforeBeamWindowCut.ipynb. Keeps the
        # clusters already identified as in-spill above (clu_beam_window_ids:
        # bridged flash time inside [BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US]),
        # so everything downstream -- efficiency, purity, 1-to-1 and 1-to-many
        # matching, metadata, every plot -- sees the neutrino-dominated in-spill
        # reco population only.
        #
        # Applied FIRST, before the other reco cuts and before
        # reassign_cluster_ID_reco: clu_beam_window_ids lives in
        # clustering-global's REAL_CLUSTER_ID namespace, which is column 3 here,
        # and reassign_cluster_ID_reco relabels clusters by avg-X and destroys
        # that namespace. Filtering after it would silently match nothing.
        #
        # RECO-side only, by design -- see the header cell: "in beam window" is
        # not a truth quantity, so the true side keeps its cosmic clusters and a
        # cosmic true cluster going unmatched here means "no in-spill reco
        # cluster near it".
        if Apply_beam_window_cut:
            n_reco_points_before_beam   = len(predicted_points)
            n_reco_clusters_before_beam = len(np.unique(predicted_points[:, 3])) if n_reco_points_before_beam else 0
            beam_ids_array   = np.fromiter(clu_beam_window_ids, dtype=float, count=len(clu_beam_window_ids))
            predicted_points = predicted_points[np.isin(predicted_points[:, 3], beam_ids_array)]
            print(f"  Event {evt}: beam-window cut kept "
                  f"{len(clu_beam_window_ids)}/{n_reco_clusters_before_beam} reco clusters, "
                  f"{len(predicted_points)}/{n_reco_points_before_beam} reco points")

        if Apply_min_reco_points_cutoff:
            predicted_points = apply_min_reco_points_cutoff(predicted_points, min_reco_points_cutoff)
        if Apply_wire_readout_sensitive_xz_plane_cut:
            predicted_points = apply_wire_readout_sensitive_yz_plane_cut_reco(predicted_points, x_min, x_max, y_min, y_max, z_min, z_max)
        # An event can legitimately end up with NO in-spill reco cluster once the
        # beam-window cut is on. That event is kept rather than skipped -- its true
        # clusters are genuine reconstruction failures and belong in the efficiency
        # denominator -- but reassign_cluster_ID_reco cannot take an empty array
        # (it concatenates per-cluster blocks), so short-circuit to an empty dict.
        # EvaluateEfficiency then marks every true cluster unmatched (reco id 8888),
        # which is the correct description of the event.
        if len(predicted_points) == 0:
            clusters_reco = {}
            print(f"  Event {evt}: no reco cluster survives the beam-window cut -- "
                  f"all true clusters counted as unmatched")
        else:
            predicted_points = reassign_cluster_ID_reco(predicted_points)
            clusters_reco    = GroupClustersByID(predicted_points)

        # ------------------------------------------------------------------
        # CLUSTER CATEGORY, EFFICIENCY, PURITY (existing functions, unchanged) --
        # all computed on the clustering-level (post-CLM) clusters_true/clusters_reco above.
        # ------------------------------------------------------------------
        cluster_category_results = cluster_category(clusters_true, output_dir=None, event=evt, apa="Combined", file_name=input_file_name)

        efficiency_results = EvaluateEfficiency(clusters_true, clusters_reco, event_key, radius_efficiency, min_recopoints_threshold)
        purity_results      = EvaluatePurity(clusters_true, clusters_reco, event_key, radius_purity_xz, radius_purity_yz, radius_purity_xy)

        # ------------------------------------------------------------------
        # METADATA + 1-TO-1 / 1-TO-MANY MATCHING (existing functions, unchanged)
        # ------------------------------------------------------------------
        event_metadata_list = add_metadata_true_clusters(
            efficiency_results, cluster_category_results,
            file_name=input_file_name, event=evt, apa="Combined", view=view, event_key=event_key)

        event_matched_pairs = MatchTrueToReco1to1(efficiency_results, purity_results)
        event_pair_metadata_list = add_metadata_true_reco_pair_cluster(
            event_matched_pairs, cluster_category_results,
            file_name=input_file_name, event=evt, apa="Combined", view=view, event_key=event_key)

        matched_true_reco_clusters = MatchTruetoReco_OneToMany(purity_results, efficiency_results)

        # ------------------------------------------------------------------
        # NEUTRINO/COSMIC LABEL RECORDS (event level): cluster-level
        # is_neutrino, for DrawLabelsAggregated below (see header note on
        # why DrawLabels' own strict q_true==1 check can misclassify once
        # q_true is a multi-valued neutrino index).
        #
        # NO beam-window flag is computed for true clusters. True clusters carry
        # no flash and no time, so it could only be inferred by matching to a
        # beam-window-flashed reco cluster -- mixing beam timing with
        # reconstruction efficiency while reading as a truth-level selection.
        # Beam window stays a RECO-side quantity (event_reco_beam_window_record
        # above / reco_cluster_info.txt).
        # ------------------------------------------------------------------
        event_cluster_type_records = build_true_cluster_type_records(
            clusters_true, input_file_name, evt, event_key
        )

        # ------------------------------------------------------------------
        # TRUE NEUTRINO INTERACTION VERTICES (mc.json), joined to their true
        # cluster by nu_idx (cluster_id = 99990+nu_idx -- an exact key, no
        # spatial matching). vertex_in_volume uses the wire-readout sensitive
        # box, the same bounds as the fiducial cut; the job-level scatter plot
        # is there to decide whether that is the right volume to keep.
        # Energies on these records: cluster_energy_MeV is the sed-derived true
        # cluster energy used by every cut/plot in this pipeline; mc.json's
        # Etot/Edep ride along as reference only and feed nothing.
        # ------------------------------------------------------------------
        event_vertex_records = build_neutrino_vertex_records(
            flatten_mc_tree(mc_tree), clusters_true, input_file_name, evt, event_key,
            x_min=x_min, x_max=x_max, y_min=y_min, y_max=y_max, z_min=z_min, z_max=z_max,
            clusters_true_precut=clusters_true_precut, min_cluster_energy=min_cluster_energy)

        # ------------------------------------------------------------------
        # EVENT-LEVEL RENDERING, ONCE PER POPULATION
        # ------------------------------------------------------------------
        # Everything above ran ONCE, against the full true and reco populations.
        # render_event_outputs (see the cell above) turns those finished records
        # into plots and text tables; calling it three times with a filtered copy
        # of the same records is what fills the three roots. Nothing is
        # re-evaluated -- an efficiency or purity value in the In root is the same
        # number as in the All root, because it IS the same number.
        #
        # Only the TRUE side is filtered: clusters_reco is passed whole to every
        # population, so a neutrino's purity still measures the cosmic
        # contamination of the reco cluster it matched.
        #
        # The In/Out event directories are created only when the event actually
        # has a neutrino of that kind, so the tree does not fill with empty ones.
        # Each root is then split a second time, by interaction channel, into its
        # own By_Interaction_Channel/ -- see inside the loop.
        event_volume_map  = build_neutrino_volume_map(event_vertex_records)
        event_channel_map = build_neutrino_channel_map(event_vertex_records)

        for population in POPULATIONS:
            pop_key   = population['key']
            pop_label = population['label']

            pop_efficiency_results = filter_records_by_label(efficiency_results, event_volume_map, pop_key)
            pop_purity_results     = filter_records_by_label(purity_results, event_volume_map, pop_key)
            pop_metadata_list      = filter_records_by_label(event_metadata_list, event_volume_map, pop_key)
            pop_pair_metadata_list = filter_records_by_label(event_pair_metadata_list, event_volume_map, pop_key)
            pop_matched_true_reco  = filter_records_by_label(matched_true_reco_clusters, event_volume_map, pop_key)
            pop_cluster_type_records = filter_records_by_label(event_cluster_type_records, event_volume_map,
                                                               pop_key, id_key='cluster_id')
            pop_vertex_records     = filter_records_by_label(event_vertex_records, event_volume_map,
                                                             pop_key, id_key='cluster_id')

            # The true clusters this population owns; cluster_category_results is
            # narrowed the same way, since DrawEfficiencyVsTrueEnergyPerEvent
            # reads its categories from there rather than from the metadata.
            if pop_key == 'all':
                pop_clusters_true = clusters_true
                pop_category_results = cluster_category_results
            else:
                pop_true_ids = {(event_key, cid) for cid in clusters_true
                                if event_volume_map.get((event_key, cid)) == pop_key}
                pop_clusters_true = {cid: pts for cid, pts in clusters_true.items()
                                     if (event_key, cid) in pop_true_ids}
                pop_category_results = {cid: data for cid, data in cluster_category_results.items()
                                        if (event_key, cid) in pop_true_ids}
                # Nothing of this kind in this event -- skip it rather than
                # creating a directory of empty plots.
                if not pop_clusters_true and not pop_vertex_records:
                    continue

            pop_event_dir = (file_output_dirs[pop_key] / f"event_{evt:03d}")

            render_start = time.time()
            render_event_outputs(
                pop_event_dir, evt, "Combined", input_file_name,
                pop_clusters_true, clusters_reco,
                pop_efficiency_results, pop_purity_results,
                pop_metadata_list, pop_pair_metadata_list,
                pop_matched_true_reco, pop_category_results,
                pop_cluster_type_records, pop_vertex_records,
                clusters_true_full=clusters_true, clusters_reco_full=clusters_reco,
                # reco-side record: All root only
                reco_beam_window_record=event_reco_beam_window_record if pop_key == 'all' else None,
                population_label=pop_label, draw=b_draw_event_level_plots)
            population_render_seconds[pop_key] += time.time() - render_start

            # --------------------------------------------------------------
            # THE SAME POPULATION, SPLIT BY INTERACTION CHANNEL
            # --------------------------------------------------------------
            # Each root gets its own By_Interaction_Channel/{numu_CC,nue_CC,NC}/,
            # so the two splits COMPOSE: the All root's channels are every true
            # neutrino, the In root's are the in-volume ones only, the Out root's
            # the out-of-volume ones. channel_map_for_population does the
            # composing; everything else is the same render call as above.
            pop_channel_map = channel_map_for_population(event_channel_map, event_volume_map, pop_key)

            for channel in CHANNELS:
                channel_records = filter_population_records(
                    pop_channel_map, channel['key'],
                    efficiency_results=efficiency_results, purity_results=purity_results,
                    metadata_list=event_metadata_list, pair_metadata_list=event_pair_metadata_list,
                    cluster_type_records=event_cluster_type_records, vertex_records=event_vertex_records,
                    matched_true_reco_clusters=matched_true_reco_clusters)

                channel_true_ids = {(event_key, cid) for cid in clusters_true
                                    if pop_channel_map.get((event_key, cid)) == channel['key']}
                if not channel_true_ids and not channel_records['vertex_records']:
                    continue

                render_start = time.time()
                render_event_outputs(
                    pop_event_dir / CHANNEL_DIRNAME / channel['dirname'],
                    evt, "Combined", input_file_name,
                    {cid: pts for cid, pts in clusters_true.items() if (event_key, cid) in channel_true_ids},
                    clusters_reco,
                    channel_records['efficiency_results'], channel_records['purity_results'],
                    channel_records['metadata_list'], channel_records['pair_metadata_list'],
                    channel_records['matched_true_reco_clusters'],
                    {cid: data for cid, data in cluster_category_results.items()
                     if (event_key, cid) in channel_true_ids},
                    channel_records['cluster_type_records'], channel_records['vertex_records'],
                    clusters_true_full=clusters_true, clusters_reco_full=clusters_reco,
                    population_label=_with_population(channel['label'], pop_label),
                    draw=b_draw_event_level_plots)
                population_render_seconds[pop_key] += time.time() - render_start

        # ------------------------------------------------------------------
        # AGGREGATE TO FILE AND JOB LEVEL
        # ------------------------------------------------------------------
        file_efficiency_results.extend(efficiency_results)
        file_purity_results.extend(purity_results)
        file_metadata_list.extend(event_metadata_list)
        file_pair_metadata_list.extend(event_pair_metadata_list)

        job_efficiency_results.extend(efficiency_results)
        job_purity_results.extend(purity_results)
        job_metadata_list.extend(event_metadata_list)
        job_pair_metadata_list.extend(event_pair_metadata_list)

        file_flash_metadata_list.extend(event_flash_metadata_list)
        job_flash_metadata_list.extend(event_flash_metadata_list)
        file_img_cluster_flash_records.extend(event_img_cluster_flash_records)
        job_img_cluster_flash_records.extend(event_img_cluster_flash_records)

        file_cluster_type_records.extend(event_cluster_type_records)
        job_cluster_type_records.extend(event_cluster_type_records)
        file_vertex_records.extend(event_vertex_records)
        job_vertex_records.extend(event_vertex_records)
        file_reco_beam_window_records.append(event_reco_beam_window_record)
        job_reco_beam_window_records.append(event_reco_beam_window_record)

        n_neutrino_points       = int(np.sum(true_points[:, 4] > 0))
        mc_records              = flatten_mc_tree(mc_tree)
        interaction_vertices    = [(r['particle'], r['energy_MeV']) for r in mc_records if r['is_interaction_vertex']]

        print(
            f"  Event {evt}: "
            f"flashes matched to clusters={len(event_flash_metadata_list)} (of {len(op_data['op_t'])} total flashes), "
            f"clusters in beam window={len(clusters_in_beam_window)}, "
            f"clustering clusters in beam window={len(clusters_clu_in_beam_window)}, "
            f"true clusters={len(clusters_true)}, reco clusters={len(clusters_reco)}, "
            f"efficiency records={len(efficiency_results)}, purity records={len(purity_results)}, 1-to-1 pairs={len(event_pair_metadata_list)}, "
            f"neutrino points={n_neutrino_points}, "
            f"interaction vertices={interaction_vertices}"
        )
        total_events_processed += 1

    # ========================================================================
    # FILE-LEVEL AGGREGATION -- once per population
    # ========================================================================
    # Same three-way split as the event level above, from this file's accumulated
    # records: render_summary_outputs writes the neutrino selection tables and
    # plots first (they describe the true population, so they are written whether
    # or not this file produced an efficiency/purity record), then the
    # efficiency/purity/matching set.
    print(f"\n  FILE-LEVEL AGGREGATION ({input_file_name}): "
          f"{len(file_efficiency_results)} efficiency, {len(file_purity_results)} purity, "
          f"{len(file_pair_metadata_list)} 1-to-1 pairs")

    file_volume_map  = build_neutrino_volume_map(file_vertex_records)
    file_channel_map = build_neutrino_channel_map(file_vertex_records)

    for population in POPULATIONS:
        pop_key   = population['key']
        pop_label = population['label']

        pop_efficiency_results   = filter_records_by_label(file_efficiency_results, file_volume_map, pop_key)
        pop_purity_results       = filter_records_by_label(file_purity_results, file_volume_map, pop_key)
        pop_metadata_list        = filter_records_by_label(file_metadata_list, file_volume_map, pop_key)
        pop_pair_metadata_list   = filter_records_by_label(file_pair_metadata_list, file_volume_map, pop_key)
        pop_cluster_type_records = filter_records_by_label(file_cluster_type_records, file_volume_map,
                                                           pop_key, id_key='cluster_id')
        pop_vertex_records       = filter_records_by_label(file_vertex_records, file_volume_map,
                                                           pop_key, id_key='cluster_id')

        if pop_key != 'all' and not (pop_metadata_list or pop_vertex_records):
            continue

        render_start = time.time()
        render_summary_outputs(
            file_output_dirs[pop_key] / "file_summary", 'file',
            pop_efficiency_results, pop_purity_results,
            pop_metadata_list, pop_pair_metadata_list,
            pop_cluster_type_records, pop_vertex_records,
            apa="Combined", file_name=input_file_name,
            population_label=pop_label, draw=b_draw_file_level_plots,
            include_cosmic_plots=population['include_cosmic_plots'])
        population_render_seconds[pop_key] += time.time() - render_start

        # This root, split again by interaction channel, into its own
        # <root>/<file>/file_summary/By_Interaction_Channel/<channel>/.
        pop_channel_map = channel_map_for_population(file_channel_map, file_volume_map, pop_key)
        for channel in CHANNELS:
            channel_records = filter_population_records(
                pop_channel_map, channel['key'],
                efficiency_results=file_efficiency_results, purity_results=file_purity_results,
                metadata_list=file_metadata_list, pair_metadata_list=file_pair_metadata_list,
                cluster_type_records=file_cluster_type_records, vertex_records=file_vertex_records)
            if not (channel_records['metadata_list'] or channel_records['vertex_records']):
                continue

            render_start = time.time()
            render_summary_outputs(
                file_output_dirs[pop_key] / "file_summary" / CHANNEL_DIRNAME / channel['dirname'], 'file',
                channel_records['efficiency_results'], channel_records['purity_results'],
                channel_records['metadata_list'], channel_records['pair_metadata_list'],
                channel_records['cluster_type_records'], channel_records['vertex_records'],
                apa="Combined", file_name=input_file_name,
                population_label=_with_population(channel['label'], pop_label),
                draw=b_draw_file_level_plots,
                # neutrinos only, so the cosmic track-geometry bar would be empty
                include_cosmic_plots=False)
            population_render_seconds[pop_key] += time.time() - render_start

    if file_flash_metadata_list:
        print(f"  FILE-LEVEL AGGREGATION ({input_file_name}): {len(file_flash_metadata_list)} flash-matched clusters")
        file_imaging_details_flashes_dir = file_output_dir / "file_summary" / "imaging_details" / "flashes"
        file_imaging_details_flashes_dir.mkdir(parents=True, exist_ok=True)
        if b_draw_file_level_plots:
            draw_flashes(file_flash_metadata_list, file_imaging_details_flashes_dir, "Combined", "File Level", "file", file_name=input_file_name)
            plt.close('all')

    if file_img_cluster_flash_records:
        file_clustering_details_flashes_dir = file_output_dir / "file_summary" / "clustering_details" / "flashes"
        file_clustering_details_flashes_dir.mkdir(parents=True, exist_ok=True)
        if b_draw_file_level_plots:
            draw_clustering_flashes(file_img_cluster_flash_records, file_clustering_details_flashes_dir, "Combined", "File Level", "file", file_name=input_file_name)
            plt.close('all')

# ============================================================================
# JOB-LEVEL AGGREGATION (existing functions, unchanged)
# ============================================================================
print(f"\n{'='*70}")
print(f"JOB SUMMARY: {total_files_processed} file(s), {total_events_processed} event(s) processed")
print(f"Total efficiency results: {len(job_efficiency_results)}")
print(f"Total purity results: {len(job_purity_results)}")
print(f"Total 1-to-1 true-reco pairs: {len(job_pair_metadata_list)}")
print(f"Total flash-matched clusters: {len(job_flash_metadata_list)}")
print(f"{'='*70}")

job_output_dirs = {key: root / "job_summary" for key, root in population_dirs.items()}
job_output_dir  = job_output_dirs['all']
job_output_dir.mkdir(parents=True, exist_ok=True)

# ============================================================================
# TRUE NEUTRINO SELECTION OUTPUTS (job level): the same six drawers and two
# writers already run at event and file level above, now aggregated over every
# file and event in the job:
#   true_neutrino_info.txt, removed_true_neutrino_info.txt,
#   labels_aggregated_job_*, labels_by_nu_idx_job_*, true_neutrino_vertices_job_*,
#   true_neutrino_vertex_volume_job_*, true_neutrino_flavor_in_volume_job_*,
#   true_neutrino_breakdown_job_*
#
# SelectionAnalysis.ipynb writes the same filenames into its own job_summary/ from
# the same functions. That duplication is intended, not an oversight: these describe
# the true population that THIS notebook's efficiency and purity numbers were
# computed on, so they belong beside them instead of in another notebook's output
# tree. Both notebooks run the same cuts on the same input, so the two copies agree;
# if they ever disagree, the cut configurations have drifted apart and that is worth
# knowing.
#
# No file_name argument at this level -- a job spans every file, so there is no one
# file to name (the drawers put "Job Level" in the title instead).
#
# All of it, plus the efficiency/purity/matching set and true_cluster_info.txt, is
# rendered once per population by render_summary_outputs -- the same function the
# file level uses, with the PerJob drawers instead of the PerFile ones.
# ============================================================================
job_volume_map  = build_neutrino_volume_map(job_vertex_records)
job_channel_map = build_neutrino_channel_map(job_vertex_records)

# Kept per population for the comparison plots and the top-level summary below.
# job_channel_records is {population key: {channel key: filtered records}} -- the
# two splits composed, which is what the three comparison versions read.
job_population_records = {}
job_channel_records    = {}

for population in POPULATIONS:
    pop_key   = population['key']
    pop_label = population['label']

    pop_efficiency_results   = filter_records_by_label(job_efficiency_results, job_volume_map, pop_key)
    pop_purity_results       = filter_records_by_label(job_purity_results, job_volume_map, pop_key)
    pop_metadata_list        = filter_records_by_label(job_metadata_list, job_volume_map, pop_key)
    pop_pair_metadata_list   = filter_records_by_label(job_pair_metadata_list, job_volume_map, pop_key)
    pop_cluster_type_records = filter_records_by_label(job_cluster_type_records, job_volume_map,
                                                       pop_key, id_key='cluster_id')
    pop_vertex_records       = filter_records_by_label(job_vertex_records, job_volume_map,
                                                       pop_key, id_key='cluster_id')
    job_population_records[pop_key] = {
        'efficiency_results': pop_efficiency_results,
        'purity_results':     pop_purity_results,
        'metadata_list':      pop_metadata_list,
        'pair_metadata_list': pop_pair_metadata_list,
        'vertex_records':     pop_vertex_records,
    }

    render_start = time.time()
    render_summary_outputs(
        job_output_dirs[pop_key], 'job',
        pop_efficiency_results, pop_purity_results,
        pop_metadata_list, pop_pair_metadata_list,
        pop_cluster_type_records, pop_vertex_records,
        apa="Combined", file_name=None,
        population_label=pop_label, draw=b_draw_job_level_plots,
        include_cosmic_plots=population['include_cosmic_plots'])
    population_render_seconds[pop_key] += time.time() - render_start

    print(f"  {population['dirname']}/job_summary: "
          f"{len(pop_metadata_list)} true clusters, {len(pop_pair_metadata_list)} 1-to-1 pairs, "
          f"{len(pop_vertex_records)} neutrino interactions")

    # ------------------------------------------------------------------
    # THIS ROOT, SPLIT BY INTERACTION CHANNEL (numu CC / nue CC / NC)
    # ------------------------------------------------------------------
    # Into <root>/job_summary/By_Interaction_Channel/<channel>/, one full plot
    # and table set each. Composed with the volume split, so the All root's
    # channels are every true neutrino, the In root's the in-volume ones, the
    # Out root's the out-of-volume ones.
    pop_channel_map = channel_map_for_population(job_channel_map, job_volume_map, pop_key)
    job_channel_records[pop_key] = {}

    for channel in CHANNELS:
        channel_records = filter_population_records(
            pop_channel_map, channel['key'],
            efficiency_results=job_efficiency_results, purity_results=job_purity_results,
            metadata_list=job_metadata_list, pair_metadata_list=job_pair_metadata_list,
            cluster_type_records=job_cluster_type_records, vertex_records=job_vertex_records)
        job_channel_records[pop_key][channel['key']] = channel_records

        render_start = time.time()
        render_summary_outputs(
            job_output_dirs[pop_key] / CHANNEL_DIRNAME / channel['dirname'], 'job',
            channel_records['efficiency_results'], channel_records['purity_results'],
            channel_records['metadata_list'], channel_records['pair_metadata_list'],
            channel_records['cluster_type_records'], channel_records['vertex_records'],
            apa="Combined", file_name=None,
            population_label=_with_population(channel['label'], pop_label),
            draw=b_draw_job_level_plots, include_cosmic_plots=False)
        population_render_seconds[pop_key] += time.time() - render_start

        print(f"    {population['dirname']}/{CHANNEL_DIRNAME}/{channel['dirname']}: "
              f"{len(channel_records['metadata_list'])} true clusters, "
              f"{len(channel_records['pair_metadata_list'])} 1-to-1 pairs, "
              f"{len(channel_records['vertex_records'])} neutrino interactions")

# ============================================================================
# INTERACTION CHANNEL COMPARISON (job level only)
# ============================================================================
# The three channels overlaid -- 1D efficiency and 1D purity on bins shared
# across all three -- in THREE versions, one per vertex-volume population:
#   ..._by_interaction_channel_{apa}.png             all true neutrinos
#   ..._by_interaction_channel_in_volume_{apa}.png   vertex in volume only
#   ..._by_interaction_channel_out_volume_{apa}.png  vertex out of volume only
# The all-neutrinos version mixes the volumes, and an out-of-volume cluster is
# only the part of the interaction that leaked into the active volume, so the
# other two are what to read for detector performance per channel.
#
# Job level only: an event holds one or two interactions, which is not a curve.
channel_comparison_output_dir = output_dir / "Interaction_Channel_Comparison"
if b_draw_job_level_plots:
    channel_comparison_output_dir.mkdir(parents=True, exist_ok=True)
    channel_keys = [c['key'] for c in CHANNELS]

    for population in POPULATIONS:
        pop_key   = population['key']
        pop_label = population['label']
        pop_channel_records = job_channel_records.get(pop_key, {})

        volume_suffix = {'all': '', 'in': '_in_volume', 'out': '_out_volume'}[pop_key]
        volume_title  = {'all': 'all true neutrinos',
                         'in':  'vertex in volume',
                         'out': 'vertex out of volume'}[pop_key]
        # Printed inside the plot, above the legend, so a plot that gets pulled
        # out of this directory still says which population it describes.
        volume_note   = {'all': 'All true neutrinos (both volumes)',
                         'in':  'In-volume true neutrinos',
                         'out': 'Out-of-volume true neutrinos'}[pop_key]

        efficiency_path = DrawClusterEfficiencyVsTrueEnergy_PopulationComparison(
            [(key, pop_channel_records.get(key, {}).get('pair_metadata_list', []),
                   pop_channel_records.get(key, {}).get('metadata_list', [])) for key in channel_keys],
            channel_comparison_output_dir, "Combined", level_name="Job Level",
            styles=CHANNEL_COMPARISON_STYLES,
            comparison_label=f"numu CC vs nue CC vs NC, {volume_title}",
            filename_suffix=f"by_interaction_channel{volume_suffix}",
            population_note=volume_note)
        purity_path = DrawPurityVsRecoCharge_PopulationComparison(
            [(key, pop_channel_records.get(key, {}).get('pair_metadata_list', [])) for key in channel_keys],
            channel_comparison_output_dir, "Combined", level_name="Job Level",
            styles=CHANNEL_COMPARISON_STYLES,
            comparison_label=f"numu CC vs nue CC vs NC, {volume_title}",
            filename_suffix=f"by_interaction_channel{volume_suffix}",
            population_note=volume_note)

        for _path in (efficiency_path, purity_path):
            if _path:
                print(f"Interaction channel comparison ({volume_title}) written to: {_path}")
    plt.close('all')

# ============================================================================
# IN vs OUT OF VOLUME COMPARISON (job level only)
# ============================================================================
# The one place the two neutrino populations meet on the same canvas: the
# clusteringlevel 1D efficiency curves and the 1D purity curves, in-volume against
# out-of-volume, on SHARED bins and a shared axis so the two are actually
# comparable (each population's own root bins itself to its own range).
#
# Job level only, deliberately: an individual event or file holds one or two
# neutrino interactions, which is not a curve.
comparison_output_dir = output_dir / "In_vs_Out_Volume_True_Neutrinos"
if b_draw_job_level_plots:
    comparison_output_dir.mkdir(parents=True, exist_ok=True)

    efficiency_comparison_path = DrawClusterEfficiencyVsTrueEnergy_PopulationComparison(
        [(key, job_population_records[key]['pair_metadata_list'],
               job_population_records[key]['metadata_list']) for key in ('in', 'out')],
        comparison_output_dir, "Combined", level_name="Job Level",
        population_note="All true neutrinos, split by vertex volume")
    purity_comparison_path = DrawPurityVsRecoCharge_PopulationComparison(
        [(key, job_population_records[key]['pair_metadata_list']) for key in ('in', 'out')],
        comparison_output_dir, "Combined", level_name="Job Level",
        population_note="All true neutrinos, split by vertex volume")

    for _path in (efficiency_comparison_path, purity_comparison_path):
        if _path:
            print(f"In vs out of volume comparison written to: {_path}")
    if efficiency_comparison_path is None and purity_comparison_path is None:
        print("In vs out of volume comparison: neither population has an entry, nothing drawn")
    plt.close('all')

# RECO CLUSTER INFO TEXT FILE (job level; same file also written per event above,
# via the same writeinformation.write_reco_cluster_info): one row per event listing
# how many DISTINCT clustering-global clusters have a beam-window-matched flash
# (event_reco_beam_window_record, built above from clusters_clu_in_beam_window).
# This is the reco-side proxy for "multiple neutrino-like activity in the beam
# spill" -- grouped by reco cluster + matched flash timing, NOT by true nu_idx
# (that's true_cluster_info.txt's num_neutrinos column, ground truth).
#
# It stays in the All root: a flash and a reco cluster carry no truth label, so
# there is no in/out-of-volume version of this table.
reco_cluster_info_path = write_reco_cluster_info(job_reco_beam_window_records, job_output_dir)
if reco_cluster_info_path:
    print(f"Reco cluster info written to: {reco_cluster_info_path}")

if job_flash_metadata_list:
    job_imaging_details_flashes_dir = job_output_dir / "imaging_details" / "flashes"
    job_imaging_details_flashes_dir.mkdir(parents=True, exist_ok=True)
    if b_draw_job_level_plots:
        draw_flashes(job_flash_metadata_list, job_imaging_details_flashes_dir, "Combined", "Job Level", "job")

if job_img_cluster_flash_records:
    job_clustering_details_flashes_dir = job_output_dir / "clustering_details" / "flashes"
    job_clustering_details_flashes_dir.mkdir(parents=True, exist_ok=True)
    if b_draw_job_level_plots:
        draw_clustering_flashes(job_img_cluster_flash_records, job_clustering_details_flashes_dir, "Combined", "Job Level", "job")

plt.close('all')

# ============================================================================
# JOB SUMMARY TEXT FILE -- one per population root
# ============================================================================
# The All root gets the full summary, flash statistics included. The In/Out roots
# get the same configuration header and runtime, their own population counts, and
# their own efficiency-by-energy table -- but no flash sections: those are
# reco-side and identical everywhere, so they are reported once, in All.
from datetime import timedelta

job_finish_dt = datetime.now()
job_runtime = time.time() - job_start_time

n_cathode_crossing  = sum(1 for r in job_img_cluster_flash_records if r['is_cathode_crossing'])
n_img_beam_window   = sum(1 for r in job_flash_metadata_list if BEAM_WINDOW_MIN_US <= r['flash_time'] <= BEAM_WINDOW_MAX_US)
n_clu_beam_window   = sum(1 for r in job_img_cluster_flash_records if BEAM_WINDOW_MIN_US <= r['flash_time'] <= BEAM_WINDOW_MAX_US)

summary_lines = []
summary_lines.append("=" * 80)
summary_lines.append("JOB SUMMARY")
summary_lines.append("=" * 80)
summary_lines.append(f"Generated: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append("")
summary_lines.append("Configuration:")
summary_lines.append(f"Parent directory: {PARENT_DIR}")
summary_lines.append(f"Plot base directory: {PLOTBASEDIR}")
summary_lines.append(f"Files to process: {files}")
summary_lines.append(f"Events to process: {events}")
if target_file is not None or target_event is not None or target_event_range is not None:
    summary_lines.append(f"Target file: {target_file if target_file else 'all'}")
    summary_lines.append(f"Target event: {target_event if target_event is not None else 'all'}")
    if target_event_range is not None:
        summary_lines.append(f"Target event range: event{target_event_range[0]}..event{target_event_range[1]} (inclusive)")
summary_lines.append("")
summary_lines.append("CLUSTERING-LEVEL evaluation (after charge-light-matching, CLM) is active:")
summary_lines.append("selections / cluster_category / efficiency / purity / 1-to-1 and 1-to-many")
summary_lines.append("matching / metadata / drawing all computed on sed-smear_readout (true) vs")
summary_lines.append("clustering-global (reco), plus optical-flash processing (img-global <->")
summary_lines.append("clustering-global flash bridging).")
summary_lines.append("")
summary_lines.append("Beam Window / Cathode-Crossing Parameters:")
summary_lines.append(f"  beam_window_min_us: {BEAM_WINDOW_MIN_US}")
summary_lines.append(f"  beam_window_max_us: {BEAM_WINDOW_MAX_US}")
summary_lines.append(f"  cathode_crossing_time_diff_max_us: {CATHODE_CROSSING_TIME_DIFF_MAX_US}")
summary_lines.append("")
summary_lines.append("Output roots (one evaluation, rendered three times -- only the TRUE side is")
summary_lines.append("filtered; every beam-window reco cluster is present in all three):")
for _population in POPULATIONS:
    summary_lines.append(f"  {_population['dirname']}/"
                         f"{'' if _population['label'] else '   <- this one, plus the reco-side diagnostics'}")
summary_lines.append("")
# Everything above is configuration and applies to every root: the per-population
# summaries below reuse it verbatim.
summary_config_lines = list(summary_lines)
summary_lines.append("=" * 80)
summary_lines.append("JOB-LEVEL AGGREGATION")
summary_lines.append("=" * 80)
summary_lines.append(f"Total files processed: {total_files_processed}")
summary_lines.append(f"Total events processed: {total_events_processed}")
summary_lines.append(f"Total efficiency results: {len(job_efficiency_results)}")
summary_lines.append(f"Total purity results: {len(job_purity_results)}")
summary_lines.append(f"Total 1-to-1 true-reco pairs: {len(job_pair_metadata_list)}")
summary_lines.append(f"Total img-global flash-matched cluster records: {len(job_flash_metadata_list)}")
summary_lines.append(f"Total clustering-global flash-matched cluster records: {len(job_img_cluster_flash_records)}")
summary_lines.append(f"Total cathode-crossing merges detected: {n_cathode_crossing}")
summary_lines.append(f"Total img-global cluster records in beam window: {n_img_beam_window}")
summary_lines.append(f"Total clustering-global cluster records in beam window: {n_clu_beam_window}")
summary_lines.append("")

if job_flash_metadata_list:
    all_img_times = [r['flash_time'] for r in job_flash_metadata_list]
    summary_lines.append("Flash Time Statistics (img-global level, per matched cluster record):")
    summary_lines.append(f"  Mean flash time:   {np.mean(all_img_times):.4f} us")
    summary_lines.append(f"  Median flash time: {np.median(all_img_times):.4f} us")
    summary_lines.append(f"  Min flash time:    {np.min(all_img_times):.4f} us")
    summary_lines.append(f"  Max flash time:    {np.max(all_img_times):.4f} us")
    summary_lines.append("")

if job_img_cluster_flash_records:
    all_clu_times = [r['flash_time'] for r in job_img_cluster_flash_records]
    summary_lines.append("Flash Time Statistics (clustering-global level, per matched cluster record):")
    summary_lines.append(f"  Mean flash time:   {np.mean(all_clu_times):.4f} us")
    summary_lines.append(f"  Median flash time: {np.median(all_clu_times):.4f} us")
    summary_lines.append(f"  Min flash time:    {np.min(all_clu_times):.4f} us")
    summary_lines.append(f"  Max flash time:    {np.max(all_clu_times):.4f} us")
    summary_lines.append("")

# Efficiency performance behind the efficiency_2d_1d_clusteringlevel 1D plots:
# mean efficiency below vs above 500 MeV, over the same population those plots
# use (1-to-1 pairs + unmatched true clusters at efficiency=0 -- the same two
# lists passed to DrawClusterEfficiencyVsTrueEnergyPerJob above).
efficiency_energy_records = summarize_cluster_efficiency_by_energy(
    job_pair_metadata_list, all_true_metadata_list=job_metadata_list, energy_threshold=500)
summary_lines.extend(format_cluster_efficiency_by_energy(efficiency_energy_records, energy_threshold=500))

runtime_lines = [
    "=" * 80,
    "JOB RUNTIME",
    "=" * 80,
    f"Job started at:  {datetime.fromtimestamp(job_start_time).strftime('%Y-%m-%d %H:%M:%S')}",
    f"Job finished at: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')}",
    f"Total job runtime: {timedelta(seconds=int(job_runtime))} ({job_runtime:.1f} seconds)",
    "=" * 80,
]
summary_lines.extend(runtime_lines)

with open(job_output_dir / "summary.txt", "w") as f:
    f.write("\n".join(summary_lines) + "\n")
print(f"Job summary written to: {job_output_dir / 'summary.txt'}")

# ----------------------------------------------------------------------------
# The In/Out roots' own summaries: same configuration header, their own counts
# and their own efficiency-by-energy table, computed from the same filtered
# record lists their plots were drawn from. No flash sections -- see the note at
# the top of this block.
# ----------------------------------------------------------------------------
for population in POPULATIONS:
    if population['label'] is None:
        continue
    pop_key = population['key']

    pop_metadata_list      = filter_records_by_label(job_metadata_list, job_volume_map, pop_key)
    pop_pair_metadata_list = filter_records_by_label(job_pair_metadata_list, job_volume_map, pop_key)
    pop_efficiency_results = filter_records_by_label(job_efficiency_results, job_volume_map, pop_key)
    pop_purity_results     = filter_records_by_label(job_purity_results, job_volume_map, pop_key)
    pop_vertex_records     = filter_records_by_label(job_vertex_records, job_volume_map,
                                                     pop_key, id_key='cluster_id')

    pop_lines = list(summary_config_lines)
    pop_lines.append("=" * 80)
    pop_lines.append(f"POPULATION: {population['label']}")
    pop_lines.append("=" * 80)
    pop_lines.append("True side restricted to this population; the reco side is NOT cut, so every")
    pop_lines.append("efficiency and purity below is the value computed against the full")
    pop_lines.append("beam-window reco population (cosmic contamination included).")
    pop_lines.append("Cosmic true clusters and unmatched reco clusters (true_cluster_id=8888) are")
    pop_lines.append(f"not part of this population -- see {population_dirs['all'].name}/.")
    pop_lines.append("")
    pop_lines.append(f"Total files processed: {total_files_processed}")
    pop_lines.append(f"Total events processed: {total_events_processed}")
    pop_lines.append(f"True neutrino interactions: {len(pop_vertex_records)}")
    pop_lines.append(f"True clusters (surviving cuts): {len(pop_metadata_list)}")
    pop_lines.append(f"Efficiency results: {len(pop_efficiency_results)}")
    pop_lines.append(f"Purity results: {len(pop_purity_results)}")
    pop_lines.append(f"1-to-1 true-reco pairs: {len(pop_pair_metadata_list)}")
    pop_lines.append("")
    pop_efficiency_energy_records = summarize_cluster_efficiency_by_energy(
        pop_pair_metadata_list, all_true_metadata_list=pop_metadata_list, energy_threshold=500)
    pop_lines.extend(format_cluster_efficiency_by_energy(pop_efficiency_energy_records, energy_threshold=500))
    pop_lines.extend(runtime_lines)

    pop_summary_path = job_output_dirs[pop_key] / "summary.txt"
    pop_summary_path.parent.mkdir(parents=True, exist_ok=True)
    with open(pop_summary_path, "w") as f:
        f.write("\n".join(pop_lines) + "\n")
    print(f"Job summary written to: {pop_summary_path}")

# ----------------------------------------------------------------------------
# TOP-LEVEL SUMMARY: one table covering all three populations side by side --
# what each root holds and how long each took to render -- so the cost and the
# size of the split can be read without opening three files.
#
# On the timings: the evaluation (cuts, cluster_category, EvaluateEfficiency,
# EvaluatePurity, matching, metadata) runs ONCE per event, before any population
# loop, and is charged to no population. What is measured per population is
# rendering only: drawing plots and writing tables. The three therefore do NOT
# sum to the job runtime -- the difference is the shared evaluation plus reading
# the input files, reported below as "shared".
# ----------------------------------------------------------------------------
overview_lines = list(summary_config_lines)
overview_lines.append("=" * 80)
overview_lines.append("POPULATIONS: WHAT EACH ROOT HOLDS AND WHAT IT COST")
overview_lines.append("=" * 80)
overview_lines.append("One evaluation, rendered three times. Only the TRUE side is filtered; every")
overview_lines.append("beam-window reco cluster is present in all three roots, so an efficiency or")
overview_lines.append("purity value is the same number wherever it appears.")
overview_lines.append("")
overview_lines.append(f"{'root':<30} {'true clu':>9} {'1-to-1':>7} {'nu int':>7} {'render s':>10}")
total_render_seconds = 0.0
for population in POPULATIONS:
    records = job_population_records.get(population['key'], {})
    seconds = population_render_seconds.get(population['key'], 0.0)
    total_render_seconds += seconds
    overview_lines.append(
        f"{population['dirname']:<30} "
        f"{len(records.get('metadata_list', [])):>9} "
        f"{len(records.get('pair_metadata_list', [])):>7} "
        f"{len(records.get('vertex_records', [])):>7} "
        f"{seconds:>10.1f}")
overview_lines.append("")
overview_lines.append(f"{'Total rendering':<30} {'':>9} {'':>7} {'':>7} {total_render_seconds:>10.1f}")
overview_lines.append(f"{'Shared (read + evaluate)':<30} {'':>9} {'':>7} {'':>7} "
                      f"{max(job_runtime - total_render_seconds, 0.0):>10.1f}")
overview_lines.append(f"{'Total job runtime':<30} {'':>9} {'':>7} {'':>7} {job_runtime:>10.1f}")
overview_lines.append("")
overview_lines.append("Columns: true clu = true clusters surviving the cuts; 1-to-1 = matched")
overview_lines.append("true-reco pairs; nu int = mc.json neutrino interactions; render s = wall-clock")
overview_lines.append("seconds spent drawing and writing that root (event + file + job level).")
overview_lines.append("")
overview_lines.append("=" * 80)
overview_lines.append("THE SAME NEUTRINOS, SPLIT BY INTERACTION CHANNEL")
overview_lines.append("=" * 80)
overview_lines.append("A second slice, composed with the volume one: what came out of the interaction")
overview_lines.append("rather than where it happened. A numu with a muon among its first daughters is")
overview_lines.append("numu CC, a nue with an electron is nue CC, neither is NC. Every root is split")
overview_lines.append(f"this way, at every level, into <root>/.../{CHANNEL_DIRNAME}/<channel>/.")
overview_lines.append("")
overview_lines.append(f"{'root / channel':<45} {'true clu':>9} {'1-to-1':>7} {'nu int':>7}")
for population in POPULATIONS:
    for channel in CHANNELS:
        records = job_channel_records.get(population['key'], {}).get(channel['key'], {})
        overview_lines.append(
            f"{population['dirname'] + '/' + channel['dirname']:<45} "
            f"{len(records.get('metadata_list', [])):>9} "
            f"{len(records.get('pair_metadata_list', [])):>7} "
            f"{len(records.get('vertex_records', [])):>7}")
overview_lines.append("")
overview_lines.append("Also written:")
overview_lines.append(f"  {channel_comparison_output_dir.name}/  -- numu CC vs nue CC vs NC,")
overview_lines.append("      1D efficiency and 1D purity overlaid on shared bins, in three versions:")
overview_lines.append("      all true neutrinos, vertex in volume, vertex out of volume (job level only)")
overview_lines.append(f"  {comparison_output_dir.name}/  -- in-volume vs out-of-volume")
overview_lines.append("      clusteringlevel 1D efficiency and 1D purity, overlaid on shared bins")
overview_lines.append("      (job level only)")
overview_lines.append(f"  {population_dirs['all'].name}/job_summary/  -- flash statistics, reco cluster")
overview_lines.append("      info and the imaging/clustering diagnostics, which carry no truth label")
overview_lines.append("      and so exist once only")
overview_lines.append("")
overview_lines.extend(runtime_lines)

overview_summary_path = output_dir / "summary.txt"
with open(overview_summary_path, "w") as f:
    f.write("\n".join(overview_lines) + "\n")
print(f"Overall summary written to: {overview_summary_path}")

print(f"\nJob finished at: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')} (runtime: {job_runtime:.1f}s)")


Output directory: multi_file_plots_charge_light_matching/Evaluation_After_TimeWindowCut_TrueNeutrinos/combined_apa_20260807_141949
  All_True_Clusters/   (reco-side diagnostics live here)
  In_Volume_True_Neutrinos/
  Out_Volume_True_Neutrinos/


FILE 1/12: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file0
Processing events 0 to 9

  Event 0: beam-window cut kept 0/15 reco clusters, 0/21848 reco points
  Event 0: no reco cluster survives the beam-window cut -- all true clusters counted as unmatched

Found 0 matched pairs of true and reco clusters (1-to-1)
Found 0 true clusters with matched reco clusters (1-to-many)
  Event 0: flashes matched to clusters=14 (of 50 total flashes), clusters in beam window=0, clustering clusters in beam window=0, true clusters=7, reco clusters=0, efficiency records=7, purity records=0, 1-to-1 pairs=0, neutrino points=0, interaction vertices=[('numu', 49.8)]
  Event 1: beam-window cut kept 1/13 reco clusters, 740/18105 re

/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/clusterpairmatching.py:31: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  merged_df       = pd.merge(efficiency_df, purity_df, on=['event', 'true_cluster_id', 'reco_cluster_id'])
/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/clusterpairmatching.py:60: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  merged_df       = pd.merge(purity_df, efficiency_df, on=['event', 'true_cluster_id', 'reco_cluster_id'])


  Event 109: flashes matched to clusters=9 (of 32 total flashes), clusters in beam window=1, clustering clusters in beam window=1, true clusters=5, reco clusters=1, efficiency records=5, purity records=1, 1-to-1 pairs=0, neutrino points=0, interaction vertices=[('numu', 0.0), ('numu', 7.8)]

  FILE-LEVEL AGGREGATION (file10): 101 efficiency, 6 purity, 4 1-to-1 pairs
  FILE-LEVEL AGGREGATION (file10): 159 flash-matched clusters

FILE 4/12: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file11
Processing events 110 to 117

  Event 110: beam-window cut kept 1/17 reco clusters, 29/19670 reco points

Found 0 matched pairs of true and reco clusters (1-to-1)
Found 0 true clusters with matched reco clusters (1-to-many)


/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/clusterpairmatching.py:31: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  merged_df       = pd.merge(efficiency_df, purity_df, on=['event', 'true_cluster_id', 'reco_cluster_id'])
/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/clusterpairmatching.py:60: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  merged_df       = pd.merge(purity_df, efficiency_df, on=['event', 'true_cluster_id', 'reco_cluster_id'])


  Event 110: flashes matched to clusters=10 (of 34 total flashes), clusters in beam window=1, clustering clusters in beam window=1, true clusters=7, reco clusters=1, efficiency records=7, purity records=1, 1-to-1 pairs=0, neutrino points=0, interaction vertices=[('numu', 0.0), ('numu', 14.8), ('numu', 0.0), ('numu', 0.0)]
  Event 111: beam-window cut kept 1/13 reco clusters, 3050/23839 reco points

Found 1 matched pairs of true and reco clusters (1-to-1)
Found 1 true clusters with matched reco clusters (1-to-many)
  Event 111: flashes matched to clusters=10 (of 27 total flashes), clusters in beam window=1, clustering clusters in beam window=1, true clusters=7, reco clusters=1, efficiency records=7, purity records=1, 1-to-1 pairs=1, neutrino points=6852, interaction vertices=[('numu', 0.0), ('numu', 438.0)]
  Event 112: beam-window cut kept 0/9 reco clusters, 0/36943 reco points
  Event 112: no reco cluster survives the beam-window cut -- all true clusters counted as unmatched

Found 0 

...

Found 0 matched pairs of true and reco clusters (1-to-1)
Found 0 true clusters with matched reco clusters (1-to-many)
  Event 28: flashes matched to clusters=14 (of 38 total flashes), clusters in beam window=2, clustering clusters in beam window=2, true clusters=9, reco clusters=2, efficiency records=9, purity records=2, 1-to-1 pairs=0, neutrino points=0, interaction vertices=[('numu', 75.4)]
  Event 29: beam-window cut kept 1/13 reco clusters, 1716/8532 reco points

Found 1 matched pairs of true and reco clusters (1-to-1)
Found 1 true clusters with matched reco clusters (1-to-many)
  Event 29: flashes matched to clusters=7 (of 27 total flashes), clusters in beam window=1, clustering clusters in beam window=1, true clusters=4, reco clusters=1, efficiency records=4, purity records=1, 1-to-1 pairs=1, neutrino points=7337, interaction vertices=[('numu', 440.2), ('numu', 0.0)]

  FILE-LEVEL AGGREGATION (file2): 72 efficiency, 11 purity, 7 1-to-1 pairs
  FILE-LEVEL AGGREGATION (file2): 120 

/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/clusterpairmatching.py:31: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  merged_df       = pd.merge(efficiency_df, purity_df, on=['event', 'true_cluster_id', 'reco_cluster_id'])
/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/clusterpairmatching.py:60: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  merged_df       = pd.merge(purity_df, efficiency_df, on=['event', 'true_cluster_id', 'reco_cluster_id'])


Found 0 matched pairs of true and reco clusters (1-to-1)
Found 0 true clusters with matched reco clusters (1-to-many)
  Event 36: flashes matched to clusters=21 (of 41 total flashes), clusters in beam window=1, clustering clusters in beam window=1, true clusters=14, reco clusters=1, efficiency records=14, purity records=1, 1-to-1 pairs=0, neutrino points=0, interaction vertices=[('numu', 0.0), ('numu', 13.7)]
  Event 37: beam-window cut kept 1/10 reco clusters, 19333/32704 reco points

Found 1 matched pairs of true and reco clusters (1-to-1)
Found 1 true clusters with matched reco clusters (1-to-many)
  Event 37: flashes matched to clusters=11 (of 31 total flashes), clusters in beam window=2, clustering clusters in beam window=1, true clusters=6, reco clusters=1, efficiency records=6, purity records=1, 1-to-1 pairs=1, neutrino points=52612, interaction vertices=[('nue', 3072.1), ('numu', 0.0), ('numu', 0.0)]
  Event 38: beam-window cut kept 1/11 reco clusters, 6069/18708 reco points

F

/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/clusterpairmatching.py:31: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  merged_df       = pd.merge(efficiency_df, purity_df, on=['event', 'true_cluster_id', 'reco_cluster_id'])
/Users/prabhjotsingh/Experiments/SBND/WireCell_Reconstruction/clusterpairmatching.py:60: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  merged_df       = pd.merge(purity_df, efficiency_df, on=['event', 'true_cluster_id', 'reco_cluster_id'])


  Event 44: flashes matched to clusters=10 (of 36 total flashes), clusters in beam window=1, clustering clusters in beam window=1, true clusters=6, reco clusters=1, efficiency records=6, purity records=1, 1-to-1 pairs=0, neutrino points=0, interaction vertices=[('numu', 44.4)]
  Event 45: beam-window cut kept 0/18 reco clusters, 0/45292 reco points
  Event 45: no reco cluster survives the beam-window cut -- all true clusters counted as unmatched

Found 0 matched pairs of true and reco clusters (1-to-1)
Found 0 true clusters with matched reco clusters (1-to-many)
  Event 45: flashes matched to clusters=20 (of 42 total flashes), clusters in beam window=0, clustering clusters in beam window=0, true clusters=13, reco clusters=0, efficiency records=13, purity records=0, 1-to-1 pairs=0, neutrino points=0, interaction vertices=[('numu', 0.2), ('numu', 0.0), ('numu', 0.0)]
  Event 46: beam-window cut kept 3/18 reco clusters, 2596/32316 reco points

Found 2 matched pairs of true and reco cluste